# Xử lý ảnh thiếu sáng cho phân loại biển báo GTSRB

**Bài tập lớn môn Xử lý ảnh — Nhóm 2.** Một notebook cho toàn bộ nghiên cứu: bảy phương pháp tăng sáng, ba bộ lọc khử nhiễu và hai thứ tự kết hợp. Chọn tham số trên validation, đánh giá subset và full-test, xuất bằng chứng có thể kiểm tra lại.

`reuse` dùng ba checkpoint và kết quả đối chứng đã xác minh để chạy khảo sát khử nhiễu; `fresh` huấn luyện từ đầu. Không tự huấn luyện khi thiếu checkpoint. Kết quả tái sử dụng được ghi rõ nguồn. Chỉ trạng thái `completed` cùng gói kết quả qua kiểm tra mới xác nhận hoàn tất; không gọi subset là full-test hoặc coi ảnh tối mô phỏng là camera đêm thực.

GTSRB tải từ [archive tác giả](https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/published-archive.html). Trọng số enhancer tải từ repo tác giả, khóa commit và SHA-256. Giữ giấy phép và ghi công; không coi việc tải là cấp phép lại. Zero-DCE chỉ dùng cho nghiên cứu/học thuật phi thương mại. Không cần API key trong notebook.

## 1. Mã nguồn đi kèm

Đây là notebook chính duy nhất: toàn bộ bước chuẩn bị dữ liệu, train/khôi phục, chọn tham số, đánh giá và đóng gói đều nằm trong file này. Cell mã nguồn tự tạo các module Python và cấu hình trong thư mục tạm; không cần tải riêng src/configs hoặc chạy notebook khác. GTSRB, trọng số tác giả và ZIP kết quả dùng lại được tải theo giao thức đã ghi.

Cell bên dưới chứa bản chụp mã nguồn của chính dự án, đọc được và có hash từng file. Không cần clone một nhánh GitHub chưa được công bố. Mã, ảnh đầu vào và trọng số tác giả được lưu trong `/tmp/gtsrb-clean`, tách khỏi Output công khai. Chỉ các kết quả và checkpoint huấn luyện mới được xuất ra `/kaggle/working`.

In [ ]:
import os, sys, json, hashlib, subprocess, shutil
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path('/tmp/gtsrb-clean')
PUBLIC = Path('/kaggle/working')
ROOT.mkdir(parents=True,exist_ok=True)
PUBLIC.mkdir(parents=True,exist_ok=True)

In [ ]:
PROJECT_FILES = {
  "src/__init__.py": "\"\"\"GTSRB low-light experiment pipeline.\"\"\"\n",
  "src/common.py": "import csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\n\nimport numpy as np\nfrom PIL import Image\n\n\ndef digest(path):\n    with open(path, 'rb') as stream:\n        return hashlib.file_digest(stream, 'sha256').hexdigest()\n\n\ndef config_id(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]\n\n\ndef stable_seed(seed, *parts):\n    return int.from_bytes(hashlib.sha256(json.dumps([seed, *parts]).encode()).digest()[:8], 'little')\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + '.tmp')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\\n')\n    temporary.replace(path)\n\n\ndef read_csv(path):\n    with open(path, newline='') as stream:\n        return list(csv.DictReader(stream))\n\n\ndef write_csv(path, rows):\n    if not rows:\n        raise ValueError('Cannot write an empty result table')\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + '.tmp')\n    with temporary.open('w', newline='') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef validate_image(image):\n    if image.ndim != 3 or image.shape[-1] != 3 or min(image.shape[:2]) < 1:\n        raise ValueError(f'Expected HWC RGB, got {image.shape}')\n    if not np.issubdtype(image.dtype, np.floating) or not np.isfinite(image).all():\n        raise ValueError('Expected finite floating RGB')\n    if image.min() < 0 or image.max() > 1:\n        raise ValueError('RGB must be within [0,1]')\n    return image.astype(np.float32, copy=False)\n\n\ndef read_image(path):\n    if Path(path).suffix == '.npy':\n        return validate_image(np.load(path, allow_pickle=False))\n    with Image.open(path) as image:\n        return np.asarray(image.convert('RGB'), dtype=np.float32) / 255\n\n\ndef save_array(path, image):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix('.tmp')\n    with temporary.open('wb') as stream:\n        np.save(stream, validate_image(image), allow_pickle=False)\n    temporary.replace(path)\n\n\ndef git_commit(path='.'):\n    result = subprocess.run(['git', '-C', str(path), 'rev-parse', 'HEAD'], capture_output=True, text=True)\n    return result.stdout.strip() if result.returncode == 0 else None\n\n\ndef environment():\n    import importlib.metadata\n    import torch\n    return {'date': datetime.now(timezone.utc).isoformat(), 'python': sys.version,\n            'platform': platform.platform(), 'commit': git_commit(),\n            'cuda': torch.cuda.is_available(), 'mps': torch.backends.mps.is_available(),\n            'packages': {p: importlib.metadata.version(p) for p in\n                         ['numpy', 'torch', 'torchvision', 'pillow', 'opencv-python-headless', 'scikit-image', 'scikit-learn']}}\n\n\ndef device_for(name='auto'):\n    import torch\n    if name == 'auto':\n        name = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'\n    return torch.device(name)\n\n\ndef synchronize(device):\n    import torch\n    if device.type == 'cuda':\n        torch.cuda.synchronize(device)\n    elif device.type == 'mps':\n        torch.mps.synchronize()\n",
  "src/data.py": "\"\"\"Official GTSRB annotations, track split, canonical ROI crops and audit.\"\"\"\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport shutil\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nfrom .common import digest, read_csv, read_image, stable_seed, write_csv, write_json\n\nBASE_URL = 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/'\nARCHIVES = {\n    'GTSRB_Final_Training_Images.zip': 'f33fd80ac59bff73c82d25ab499e03a3',\n    'GTSRB_Final_Test_Images.zip': 'c7e4e6327067d32654124b0fe9e82185',\n    'GTSRB_Final_Test_GT.zip': 'fe31e9c9270bbcd7b84b7f21a9d9d9e5',\n}\n\n\ndef download(root=Path('.')):\n    root = Path(root)\n    raw = root / 'data/raw/gtsrb'\n    raw.mkdir(parents=True, exist_ok=True)\n    records = []\n    for name, expected_md5 in ARCHIVES.items():\n        archive = raw / name\n        if not archive.exists():\n            print(f'Downloading {name}', flush=True)\n            temporary = archive.with_suffix('.zip.part')\n            with urllib.request.urlopen(BASE_URL + name, timeout=120) as response, temporary.open('wb') as stream:\n                shutil.copyfileobj(response, stream)\n            temporary.replace(archive)\n        with archive.open('rb') as stream:\n            actual = hashlib.file_digest(stream, 'md5').hexdigest()\n        if actual != expected_md5:\n            raise ValueError(f'Archive checksum mismatch: {archive}; move aside and download again')\n        target = raw / archive.stem\n        if not (target / '.complete').exists():\n            target.mkdir(exist_ok=True)\n            with zipfile.ZipFile(archive) as zipped:\n                for item in zipped.infolist():\n                    if not (target / item.filename).resolve().is_relative_to(target.resolve()):\n                        raise ValueError(f'Unsafe archive member: {item.filename}')\n                zipped.extractall(target)\n            (target / '.complete').write_text(digest(archive))\n        records.append({'url': BASE_URL + name, 'archive': name, 'sha256': digest(archive),\n                        'md5': actual, 'verified_at': datetime.now(timezone.utc).isoformat(),\n                        'downloaded_at': datetime.fromtimestamp(archive.stat().st_mtime, timezone.utc).isoformat()})\n    write_json(root / 'data/manifests/gtsrb_sources.json', records)\n\n\ndef split_tracks(rows, seed=42, fraction=0.2):\n    if not 0 < fraction < 1:\n        raise ValueError('validation_fraction must be between 0 and 1')\n    for label in range(43):\n        members = [r for r in rows if r['split'] in {'train', 'validation'} and int(r['class_id']) == label]\n        groups = sorted({r['group_id'] for r in members})\n        if len(groups) < 2:\n            raise ValueError(f'Class {label} needs at least two train tracks')\n        rng = np.random.default_rng(stable_seed(seed, label))\n        rng.shuffle(groups)\n        count = max(1, min(len(groups) - 1, round(len(groups) * fraction)))\n        validation = set(groups[:count])\n        for row in members:\n            row['split'] = 'validation' if row['group_id'] in validation else 'train'\n    return rows\n\n\ndef quarantine_test_overlap(rows):\n    \"\"\"Keep test intact; exclude entire training tracks with exact test copies.\"\"\"\n    test_hashes = {r['pixel_sha256'] for r in rows if r['split'] == 'test'}\n    groups = {r['group_id'] for r in rows if r['split'] != 'test' and r['pixel_sha256'] in test_hashes}\n    for row in rows:\n        if row['split'] != 'test' and row['group_id'] in groups:\n            row['split'] = 'quarantine'\n    return sorted(groups)\n\n\ndef crop_box(roi, width, height, padding=0.05, inclusive=True):\n    x1, y1, x2, y2 = map(int, roi)\n    x2 += int(inclusive)\n    y2 += int(inclusive)\n    if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height) or not 0 <= padding <= 1:\n        raise ValueError(f'Invalid ROI {roi} for {width}x{height}')\n    dx, dy = (x2 - x1) * padding, (y2 - y1) * padding\n    return [max(0, math.floor(x1-dx)), max(0, math.floor(y1-dy)),\n            min(width, math.ceil(x2+dx)), min(height, math.ceil(y2+dy))]\n\n\ndef prepare(root, config):\n    root = Path(root)\n    manifest = root / 'data/manifests/gtsrb_samples.csv'\n    if manifest.exists():\n        print('Manifest already exists; skipping prepare (use check-data, or a different --root for a new protocol)', flush=True)\n        return\n    raw = root / 'data/raw/gtsrb'\n    train_csv = sorted(raw.glob('GTSRB_Final_Training_Images/**/GT-*.csv'))\n    test_csv = list(raw.glob('GTSRB_Final_Test_GT/**/GT-final_test.csv'))\n    if len(train_csv) != 43 or len(test_csv) != 1:\n        raise ValueError('Expected 43 training annotations and one official test annotation; run download')\n    test_folder = raw / 'GTSRB_Final_Test_Images/GTSRB/Final_Test/Images'\n    rows = []\n    seen_paths = set()\n    for annotation in train_csv + test_csv:\n        test = annotation == test_csv[0]\n        with annotation.open(newline='') as stream:\n            annotations = list(csv.DictReader(stream, delimiter=';'))\n        for item in annotations:\n            filename = item['Filename']\n            if Path(filename).name != filename:\n                raise ValueError('Annotation filename must be a basename')\n            source = (test_folder if test else annotation.parent) / filename\n            label = int(item['ClassId'])\n            if not 0 <= label < 43 or (not test and int(annotation.parent.name) != label):\n                raise ValueError(f'Invalid label: {source}')\n            track_match = re.fullmatch(r'(\\d{5})_(\\d{5})\\.ppm', filename)\n            if not test and track_match is None:\n                raise ValueError(f'Unverified train track filename: {filename}')\n            source_id = f'test/{Path(filename).stem}' if test else f'train/{label:05d}/{Path(filename).stem}'\n            with Image.open(source) as image:\n                image.load()\n                width, height = image.size\n                pixels_sha = hashlib.sha256(image.convert('RGB').tobytes() + str(image.size).encode()).hexdigest()\n            if (width, height) != (int(item['Width']), int(item['Height'])):\n                raise ValueError(f'Annotation/image dimensions disagree: {source}')\n            box = crop_box([item[k] for k in ['Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2']], width, height,\n                           config['crop']['padding'], config['crop']['roi_end_inclusive'])\n            if source in seen_paths:\n                raise ValueError(f'Duplicate annotation: {source}')\n            seen_paths.add(source)\n            rows.append({'sample_id': 'gtsrb/' + source_id, 'dataset': 'gtsrb',\n                         'source_path': source.relative_to(root).as_posix(), 'source_id': source_id,\n                         'group_id': source_id if test else f'train/{label:05d}/{track_match[1]}',\n                         'bbox': json.dumps(box), 'original_label': label, 'class_id': label,\n                         'split': 'test' if test else 'train', 'width': width, 'height': height,\n                         'source_sha256': digest(source), 'pixel_sha256': pixels_sha,\n                         'crop_path': f'data/processed/gtsrb-final/{source_id}.png'})\n    all_images = set(raw.glob('GTSRB_Final_Training_Images/**/*.ppm')) | set(test_folder.glob('*.ppm'))\n    if seen_paths != all_images:\n        raise ValueError(f'Image/annotation coverage mismatch: {len(seen_paths)} vs {len(all_images)}')\n    quarantined = quarantine_test_overlap(rows)\n    write_json(root / 'data/manifests/quarantine.json', {'reason': 'Training track contains exact pixel copies of official test images; official test kept intact',\n               'group_ids': quarantined, 'sample_ids': [r['sample_id'] for r in rows if r['split'] == 'quarantine']})\n    split_tracks(rows, config['split_seed'], config['validation_fraction'])\n    duplicates = defaultdict(list)\n    for row in rows:\n        duplicates[row['pixel_sha256']].append(row['sample_id'])\n    duplicate_groups = [v for v in duplicates.values() if len(v) > 1]\n    write_json(root / 'data/manifests/duplicate_audit.json', duplicate_groups)\n    for i, row in enumerate(rows):\n        with Image.open(root / row['source_path']) as image:\n            crop = image.convert('RGB').crop(json.loads(row['bbox'])).resize((64, 64), Image.Resampling.BILINEAR)\n            target = root / row['crop_path']\n            target.parent.mkdir(parents=True, exist_ok=True)\n            crop.save(target)\n        if i % 10000 == 0:\n            print(f'Prepared {i}/{len(rows)} crops', flush=True)\n    write_csv(manifest, rows)\n    write_json(manifest.with_suffix('.json'), {'sha256': digest(manifest), 'crop': config['crop'],\n               'bbox_convention': 'padded xyxy, exclusive right/bottom; original ROI inclusive by config',\n               'training_archive': 'GTSRB_Final_Training_Images.zip',\n               'split_seed': config['split_seed'], 'validation_fraction': config['validation_fraction'],\n               'test_group_limitation': 'Official test has no verified tracks; source_id used'})\n    audit(root)\n    gallery(root, rows)\n    return rows\n\n\ndef audit(root):\n    root = Path(root)\n    manifest = root / 'data/manifests/gtsrb_samples.csv'\n    meta = json.loads(manifest.with_suffix('.json').read_text())\n    if digest(manifest) != meta['sha256']:\n        raise ValueError('Locked manifest was modified')\n    rows = read_csv(manifest)\n    ids = [r['sample_id'] for r in rows]\n    if len(ids) != len(set(ids)):\n        raise ValueError('Duplicate sample IDs')\n    split_sets = {}\n    counts = Counter()\n    near_hashes, image_stats = defaultdict(list), []\n    for split in ['train', 'validation', 'test']:\n        subset = [r for r in rows if r['split'] == split]\n        if {int(r['class_id']) for r in subset} != set(range(43)):\n            raise ValueError(f'Missing/invalid classes in {split}')\n        split_sets[split] = {r['group_id'] for r in subset}\n    if any(split_sets[a] & split_sets[b] for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]):\n        raise ValueError('Track leakage across splits')\n    sources, hashes = {}, defaultdict(set)\n    for row in rows:\n        if row['source_id'] in sources:\n            raise ValueError('Source duplicated in manifest')\n        sources[row['source_id']] = row['split']\n        source = root / row['source_path']\n        if digest(source) != row['source_sha256']:\n            raise ValueError(f'Source changed: {source}')\n        x1, y1, x2, y2 = json.loads(row['bbox'])\n        if not (0 <= x1 < x2 <= int(row['width']) and 0 <= y1 < y2 <= int(row['height'])):\n            raise ValueError('Invalid crop box')\n        crop = read_image(root / row['crop_path'])\n        if crop.shape != (64, 64, 3):\n            raise ValueError('Canonical image must be RGB 64x64')\n        with Image.open(source) as original:\n            small = np.asarray(original.convert('L').resize((9,8), Image.Resampling.BILINEAR))\n        near_hash = np.packbits(small[:,1:] > small[:,:-1]).tobytes().hex()\n        near_hashes[near_hash].append({'sample_id': row['sample_id'], 'split': row['split'], 'group_id': row['group_id']})\n        image_stats.append({'sample_id': row['sample_id'], 'split': row['split'], 'class_id': row['class_id'],\n                            'width': row['width'], 'height': row['height'], 'mean_rgb': float(crop.mean())})\n        if row['split'] != 'quarantine':\n            hashes[row['pixel_sha256']].add(row['split'])\n        counts[(row['split'], int(row['class_id']))] += 1\n    leaks = [key for key, value in hashes.items() if len(value) > 1]\n    near_candidates = [v for v in near_hashes.values() if len({r['split'] for r in v}) > 1]\n    write_json(root / 'data/manifests/near_duplicate_candidates.json', near_candidates)\n    write_csv(root / 'data/manifests/image_statistics.csv', image_stats)\n    write_json(root / 'data/manifests/audit.json', {'n_images': len(rows), 'n_quarantined': sum(r['split']=='quarantine' for r in rows), 'cross_split_duplicate_hashes': leaks,\n               'n_tracks': {s: len(v) for s, v in split_sets.items()},\n               'status': 'failed' if leaks else 'completed',\n               'visual_roi_review': 'pending; inspect roi_gallery.jpg',\n               'near_duplicates': 'Equal 64-bit dHash cross-split candidates saved for visual review; not a proof of duplication',\n               'near_duplicate_candidate_groups': len(near_candidates)})\n    write_csv(root / 'data/manifests/class_distribution.csv',\n              [{'split': s, 'class_id': c, 'count': n} for (s, c), n in sorted(counts.items())])\n    if leaks:\n        raise ValueError('Exact pixel duplicates cross splits; see audit.json; do not silently remove test images')\n    return rows\n\n\ndef gallery(root, rows):\n    # Fixed selection: first train image of each class, then seven darkest remaining train crops.\n    train = [r for r in rows if r['split'] == 'train']\n    selected = [next(r for r in train if int(r['class_id']) == c) for c in range(43)]\n    known = {r['sample_id'] for r in selected}\n    selected += sorted([r for r in train if r['sample_id'] not in known],\n                       key=lambda r: float(read_image(root / r['crop_path']).mean()))[:7]\n    sheet = Image.new('RGB', (10 * 144, 5 * 166), 'white')\n    draw = ImageDraw.Draw(sheet)\n    for i, row in enumerate(selected):\n        with Image.open(root / row['source_path']) as original:\n            original = original.convert('RGB')\n            ImageDraw.Draw(original).rectangle(json.loads(row['bbox']), outline='red', width=1)\n            scale = min(138/original.width, 138/original.height)\n            original = original.resize((round(original.width*scale), round(original.height*scale)), Image.Resampling.NEAREST)\n            x, y = (i % 10) * 144, (i // 10) * 166\n            sheet.paste(original, (x, y))\n            draw.text((x, y+140), f\"class {row['class_id']}\", fill='black')\n    folder = root / 'data/manifests'\n    sheet.save(folder / 'roi_gallery.jpg')\n    write_json(folder / 'gallery_ids.json', [r['sample_id'] for r in selected])\n\n\ndef subset_rows(rows, split, per_class=None, seed=42):\n    rows = [r for r in rows if r['split'] == split]\n    if per_class is None:\n        return rows\n    if per_class < 1:\n        raise ValueError('per_class must be positive')\n    selected = []\n    for label in range(43):\n        members = [r for r in rows if int(r['class_id']) == label]\n        # Select whole groups, possibly exceeding the requested image budget.\n        groups = sorted({r['group_id'] for r in members}, key=lambda g: stable_seed(seed, g))\n        chosen = []\n        for group in groups:\n            chosen.extend(r for r in members if r['group_id'] == group)\n            if len(chosen) >= per_class:\n                break\n        selected.extend(chosen)\n    if not selected:\n        raise ValueError(f'Empty {split} split')\n    return selected\n",
  "src/degradation.py": "import numpy as np\nfrom .common import stable_seed, validate_image\n\n\ndef degrade(image, gamma_dark=1.0, poisson_peak=None, gaussian_sigma=0.0,\n            noise_seed=101, sample_id='', condition=''):\n    image = validate_image(image)\n    values = [gamma_dark, gaussian_sigma] + ([] if poisson_peak is None else [poisson_peak])\n    if not np.isfinite(values).all() or gamma_dark <= 0 or gaussian_sigma < 0 or (poisson_peak is not None and poisson_peak <= 0):\n        raise ValueError('Invalid degradation parameters')\n    rng = np.random.default_rng(stable_seed(noise_seed, sample_id, condition))\n    result = image ** gamma_dark\n    if poisson_peak is not None:\n        result = rng.poisson(poisson_peak * result) / poisson_peak\n    if gaussian_sigma:\n        result = result + rng.normal(0, gaussian_sigma, image.shape)\n    return np.clip(result, 0, 1).astype(np.float32)\n",
  "src/enhancement.py": "\"\"\"RGB-only enhancement. No adapter accepts a reference image or label.\"\"\"\nimport argparse\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nimport yaml\n\nfrom .common import digest, git_commit, stable_seed, validate_image\nfrom .model import image_tensor\n\nDEEP = {'zero_dce', 'retinexformer', 'diffusion'}\nMETHODS = ('identity', 'gamma', 'he', 'clahe', 'retinex', 'zero_dce', 'retinexformer', 'diffusion')\n\n\ndef classical(image, method, params=None):\n    image = validate_image(image)\n    params = params or {}\n    if method == 'identity':\n        return image.copy()\n    if method == 'gamma':\n        gamma = params.get('gamma', 0.6)\n        if not 0 < gamma < 1:\n            raise ValueError('Enhancement gamma must be in (0,1)')\n        return image ** gamma\n    if method in {'he', 'clahe'}:\n        ycc = cv2.cvtColor(np.rint(image * 255).astype(np.uint8), cv2.COLOR_RGB2YCrCb)\n        if method == 'he':\n            ycc[..., 0] = cv2.equalizeHist(ycc[..., 0])\n        else:\n            clip, tiles = params.get('clip_limit', 2.0), params.get('tile_grid', [8, 8])\n            if not np.isfinite(clip) or clip <= 0 or len(tiles) != 2 or any(not isinstance(t, int) or t < 1 for t in tiles):\n                raise ValueError('Invalid CLAHE parameters')\n            ycc[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=tuple(tiles)).apply(ycc[..., 0])\n        return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2RGB).astype(np.float32) / 255\n    if method == 'retinex':\n        sigmas = params.get('sigmas', [3, 10, 20])\n        if not sigmas or not np.isfinite(sigmas).all() or min(sigmas) <= 0:\n            raise ValueError('MSR sigmas must be finite and positive')\n        result = sum(np.log(image + 1e-6) - np.log(cv2.GaussianBlur(image, (0, 0), sigmaX=s) + 1e-6)\n                     for s in sigmas) / len(sigmas)\n        lo, hi = np.percentile(result, [1, 99])\n        return image.copy() if hi - lo < 1e-6 else np.clip((result-lo)/(hi-lo), 0, 1).astype(np.float32)\n    raise ValueError(f'Unknown classical enhancer: {method}')\n\n\ndef import_file(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef namespace(value):\n    return argparse.Namespace(**{k: namespace(v) if isinstance(v, dict) else v for k, v in value.items()})\n\n\ndef provenance(method, params):\n    metadata = {'method': method, 'params': params, 'precision': 'float32', 'input': 'RGB [0,1]',\n                'reference_access': False, 'output_storage': 'float32 npy; no further quantization'}\n    if method in DEEP:\n        checkpoint, repo = Path(params['checkpoint']), Path(params['repo'])\n        if not checkpoint.is_file():\n            raise FileNotFoundError(f'{method}: missing author checkpoint {checkpoint}; see README.md')\n        commit = git_commit(repo)\n        if commit is None:\n            raise ValueError(f'{method}: author repository must have a recorded git commit')\n        metadata.update(checkpoint_sha256=digest(checkpoint), repo_commit=commit)\n        code_files = {'zero_dce': ['Zero-DCE_code/model.py'],\n                      'retinexformer': ['basicsr/models/archs/RetinexFormer_arch.py'],\n                      'diffusion': ['models/ddm.py', 'models/unet.py', 'models/mods.py', 'models/wavelet.py']}[method]\n        metadata['code_sha256'] = {f: digest(repo / f) for f in code_files}\n        if 'config' in params:\n            metadata['config_sha256'] = digest(params['config'])\n    return metadata\n\n\nclass Enhancer:\n    def __init__(self, method, params, device):\n        if method not in METHODS:\n            raise ValueError(f'Unknown method: {method}')\n        self.method, self.params, self.device = method, params, device\n        self.metadata = provenance(method, params)\n        self.model = None\n        if method not in DEEP:\n            return\n        repo = Path(params['repo']).resolve()\n        # Load only author architectures, avoiding their dataset/GT-dependent evaluation scripts.\n        if method == 'zero_dce':\n            module = import_file('author_zero_dce', repo / 'Zero-DCE_code/model.py')\n            self.model = module.enhance_net_nopool()\n        elif method == 'retinexformer':\n            module = import_file('author_retinexformer', repo / 'basicsr/models/archs/RetinexFormer_arch.py')\n            cfg = yaml.safe_load(Path(params['config']).read_text())['network_g'].copy()\n            if cfg.pop('type') != 'RetinexFormer':\n                raise ValueError('Expected RetinexFormer architecture config')\n            self.model = module.RetinexFormer(**cfg)\n        else:\n            # Author modules use absolute imports (models, utils); do not import other repos that way.\n            sys.path.insert(0, str(repo))\n            from models.ddm import Net\n            cfg = namespace(yaml.safe_load(Path(params['config']).read_text()))\n            cfg.device = device\n            steps = params['sampling_timesteps']\n            if not isinstance(steps, int) or not 1 <= steps <= cfg.diffusion.num_diffusion_timesteps or cfg.diffusion.num_diffusion_timesteps % steps:\n                raise ValueError('Sampling steps must divide the author diffusion timestep count')\n            self.model = Net(argparse.Namespace(sampling_timesteps=steps), cfg)\n        # Legacy diffusion checkpoint contains argparse.Namespace; allow only that safe container.\n        with torch.serialization.safe_globals([argparse.Namespace]):\n            state = torch.load(params['checkpoint'], map_location='cpu', weights_only=True)\n        if method == 'retinexformer':\n            state = state['params']\n        elif method == 'diffusion':\n            state = state['ema_helper'] if params.get('ema', True) else state['state_dict']\n        state = {k.removeprefix('module.'): v for k, v in state.items()}\n        self.model.load_state_dict(state, strict=True)\n        self.model.to(device).eval()\n        self.metadata['padding'] = 'replicate to multiple of 32' if method == 'diffusion' else 'replicate to multiple of 4' if method == 'retinexformer' else 'none'\n\n    @torch.inference_mode()\n    def __call__(self, image, sample_id=''):\n        image = validate_image(image)\n        if self.model is None:\n            return validate_image(classical(image, self.method, self.params))\n        tensor = image_tensor(image).unsqueeze(0).to(self.device)\n        h, w = image.shape[:2]\n        multiple = 32 if self.method == 'diffusion' else 4 if self.method == 'retinexformer' else 1\n        tensor = F.pad(tensor, (0, (-w) % multiple, 0, (-h) % multiple), mode='replicate')\n        if self.method == 'diffusion':\n            seed = stable_seed(self.params.get('inference_seed', 230), sample_id) % (2**63-1)\n            torch.manual_seed(seed)\n        output = self.model(tensor)\n        if self.method == 'zero_dce':\n            output = output[1]\n        elif self.method == 'diffusion':\n            output = output['pred_x']\n        if output.shape[:2] != (1, 3) or output.shape[2:] != tensor.shape[2:] or not torch.isfinite(output).all():\n            raise ValueError(f'{self.method}: malformed/nonfinite output')\n        result = output[0, :, :h, :w].clamp(0, 1).cpu().numpy().transpose(1, 2, 0)\n        return validate_image(result)\n\n\ndef validation_grid(method, defaults):\n    if method == 'gamma':\n        return [{'gamma': g} for g in [0.4, 0.6, 0.8]]\n    if method == 'clahe':\n        return [{'clip_limit': c, 'tile_grid': t} for c in [1.0, 2.0, 4.0] for t in [[4, 4], [8, 8]]]\n    if method == 'retinex':\n        return [{'sigmas': s} for s in [[3, 10, 20], [5, 15, 30]]]\n    return [defaults]\n",
  "src/experiment.py": "\"\"\"Shared float cache, validation search, frozen protocol and paired evaluation.\"\"\"\nfrom copy import deepcopy\nimport json\nfrom pathlib import Path\nimport time\nimport shutil\n\nimport numpy as np\nimport torch\n\nfrom .common import (config_id, digest, environment, read_csv, read_image, save_array,\n                     synchronize, write_csv, write_json)\nfrom .data import subset_rows\nfrom .degradation import degrade\nfrom .enhancement import Enhancer, METHODS, provenance, validation_grid\nfrom .metrics import classification, image_metrics\nfrom .model import make_cnn, image_tensor, cnn_config\n\n\ndef source_code_hash():\n    return config_id({p.name: digest(p) for p in sorted(Path(__file__).parent.glob('*.py'))})\n\n\ndef release_image_cache(root, cached):\n    \"\"\"Remove only a completed, generated cache directory after every CNN used it.\"\"\"\n    if not cached:\n        return\n    base = (Path(root) / 'data/cache').resolve()\n    folders = {(Path(root) / r['path']).resolve().parent for r in cached}\n    if len(folders) != 1:\n        raise ValueError('Refusing to remove mixed cache directories')\n    folder = folders.pop()\n    if folder.parent != base or len(folder.name) != 16 or any(c not in '0123456789abcdef' for c in folder.name):\n        raise ValueError('Refusing to remove a path outside the generated cache')\n    metadata = json.loads((folder / 'metadata.json').read_text())\n    if metadata.get('status') != 'completed' or metadata['sample_ids'] != [r['sample_id'] for r in cached]:\n        raise ValueError('Cache metadata does not match the completed samples')\n    shutil.rmtree(folder)\n\n\ndef manifest_rows(root):\n    manifest = Path(root) / 'data/manifests/gtsrb_samples.csv'\n    if digest(manifest) != json.loads(manifest.with_suffix('.json').read_text())['sha256']:\n        raise ValueError('Manifest has changed')\n    audit_path = manifest.parent / 'audit.json'\n    if not audit_path.exists() or json.loads(audit_path.read_text())['status'] != 'completed':\n        raise ValueError('Data audit is missing or failed; run check-data')\n    return read_csv(manifest), digest(manifest)\n\n\ndef cache_images(root, rows, manifest_hash, config, condition, noise_seed, method, params, device):\n    \"\"\"Cache each output once across CNN seeds. Never accept partially complete runs.\"\"\"\n    degradation = {} if condition == 'clean' else config['conditions'][condition]\n    metadata = {'manifest_sha256': manifest_hash, 'code_hash': source_code_hash(),\n                'condition': condition, 'noise_seed': noise_seed, 'degradation': degradation,\n                'enhancer': {'method': 'clean'} if method == 'clean' else provenance(method, params),\n                'device': str(device), 'ssim': config['ssim'],\n                'sample_ids': [r['sample_id'] for r in rows]}\n    folder = Path(root) / 'data/cache' / config_id(metadata)\n    index = folder / 'images.csv'\n    if index.exists():\n        cached = read_csv(index)\n        if [r['sample_id'] for r in cached] != metadata['sample_ids']:\n            raise ValueError('Cache sample identity mismatch')\n        if any(not (Path(root) / r['path']).is_file() or digest(Path(root) / r['path']) != r['sha256'] for r in cached):\n            raise ValueError('Cache missing or modified; move this cache folder aside and rerun')\n        return cached\n    write_json(folder / 'metadata.json', dict(metadata, status='running', environment=environment()))\n    enhancer = None if method == 'clean' else Enhancer(method, params, device)\n    warmup_ms = 0.0\n    if enhancer is not None:\n        image = degrade(read_image(Path(root) / rows[0]['crop_path']), **degradation,\n                        noise_seed=noise_seed, sample_id=rows[0]['sample_id'], condition=condition)\n        synchronize(device)\n        start = time.perf_counter()\n        for _ in range(2):\n            enhancer(image, rows[0]['sample_id'])\n        synchronize(device)\n        warmup_ms = (time.perf_counter()-start)*1000\n    if device.type == 'cuda':\n        torch.cuda.reset_peak_memory_stats(device)\n    records = []\n    try:\n        for row in rows:\n            reference = read_image(Path(root) / row['crop_path'])\n            image = reference if method == 'clean' else degrade(reference, **degradation, noise_seed=noise_seed,\n                                                               sample_id=row['sample_id'], condition=condition)\n            synchronize(device)\n            start = time.perf_counter()\n            output = reference if enhancer is None else enhancer(image, row['sample_id'])\n            synchronize(device)\n            latency = 0.0 if enhancer is None else (time.perf_counter()-start)*1000\n            if output.shape != reference.shape:\n                raise ValueError('Enhancer changed canonical image dimensions')\n            path = folder / (config_id(row['sample_id']) + '.npy')\n            save_array(path, output)\n            records.append({'sample_id': row['sample_id'], 'condition': condition, 'noise_seed': noise_seed,\n                            **{k: degradation.get(k, '') for k in ['gamma_dark', 'poisson_peak', 'gaussian_sigma']},\n                            'method': method, 'config_id': config_id(params),\n                            'path': path.relative_to(root).as_posix(), 'sha256': digest(path),\n                            'latency_ms': latency, 'input_mean': float(image.mean()),\n                            'output_mean': float(output.mean()),\n                            'input_clipped_fraction': float(np.mean((image == 0) | (image == 1))),\n                            **image_metrics(reference, output, config['ssim'])})\n        write_csv(index, records)\n        write_json(folder / 'metadata.json', dict(metadata, status='completed', environment=environment(),\n                   warmup_ms=warmup_ms, batch_size=1, resolution=[64, 64], precision='float32',\n                   latency_includes='tensor transfer and synchronized inference; excludes file I/O and metrics',\n                   peak_vram_bytes=torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else None))\n    except Exception as error:\n        write_json(folder / 'metadata.json', dict(metadata, status='failed', error=str(error)))\n        raise\n    return records\n\n\ndef load_cnn(checkpoint, manifest_hash, device, config=None):\n    saved = torch.load(checkpoint, map_location=device, weights_only=True)\n    if saved['manifest_sha256'] != manifest_hash:\n        raise ValueError('CNN trained on a different manifest')\n    cfg = cnn_config(saved.get('cnn_config'))\n    if config is not None and cfg != cnn_config(config):\n        raise ValueError('Checkpoint classifier config does not match experiment config')\n    model = make_cnn(cfg, pretrained=False).to(device)\n    model.load_state_dict(saved['state_dict'])\n    return model.eval(), saved\n\n\n@torch.inference_mode()\ndef classifier_latency(model, images, device):\n    \"\"\"Measure resize + normalization + classifier; exclude transfer and file I/O.\"\"\"\n    for _ in range(2):\n        model(images)\n    synchronize(device)\n    start = time.perf_counter()\n    for _ in range(5):\n        model(images)\n    synchronize(device)\n    return {'latency_ms_per_image': 1000 * (time.perf_counter() - start) / (5 * len(images)),\n            'batch_size': len(images), 'crop_resolution': list(images.shape[-2:]),\n            'device': str(device), 'warmup_batches': 2, 'measured_batches': 5,\n            'includes': 'resize, normalization, classifier; excludes transfer, I/O, enhancer'}\n\n\n@torch.inference_mode()\ndef predict(root, rows, cached, model, config, device):\n    if [r['sample_id'] for r in rows] != [r['sample_id'] for r in cached]:\n        raise ValueError('Input/output sample IDs do not match')\n    predictions = []\n    batch_size = config['cnn']['batch_size']\n    for start in range(0, len(rows), batch_size):\n        batch = cached[start:start+batch_size]\n        images = torch.stack([image_tensor(read_image(Path(root) / r['path'])) for r in batch]).to(device)\n        probabilities = model(images).softmax(1).cpu().numpy()\n        if not np.isfinite(probabilities).all():\n            raise ValueError('Nonfinite CNN predictions')\n        for row, output, probabilities_i in zip(rows[start:start+batch_size], batch, probabilities):\n            predictions.append({'sample_id': row['sample_id'], 'group_id': row['group_id'],\n                                'source_id': row['source_id'], 'class_id': int(row['class_id']),\n                                'prediction': int(probabilities_i.argmax()),\n                                'true_probability': float(probabilities_i[int(row['class_id'])]),\n                                'noise_seed': output['noise_seed'], 'psnr': float(output['psnr']),\n                                'ssim': float(output['ssim']), 'latency_ms': float(output['latency_ms']),\n                                'output_path': output['path']})\n    return predictions\n\n\ndef choose_pipeline(scores, tolerance=0.005):\n    if not scores:\n        raise ValueError('No validation scores')\n    best = max(r['macro_f1'] for r in scores)\n    eligible = [r for r in scores if best - r['macro_f1'] < tolerance or r['macro_f1'] == best]\n    return min(eligible, key=lambda r: (r['latency_ms'], -r['macro_f1'], r['method']))\n\n\ndef tune(root, config, checkpoints, device, output, per_class=None, methods=METHODS, release_cache=False):\n    root, output = Path(root), Path(output)\n    if (output / 'selection.json').exists():\n        raise FileExistsError('Selection already exists; use a new output directory')\n    if 'identity' not in methods or len(set(methods)) != len(methods):\n        raise ValueError('Validation must include identity and unique methods')\n    all_rows, manifest_hash = manifest_rows(root)\n    rows = subset_rows(all_rows, 'validation', per_class)\n    models = [load_cnn(p, manifest_hash, device, config['cnn']) for p in checkpoints]\n    seeds = [s['seed'] for _, s in models]\n    if len(seeds) != len(set(seeds)):\n        raise ValueError('Duplicate CNN training seeds')\n    scores, details, selected_params = [], [], {}\n    for method in methods:\n        candidates = []\n        for params in validation_grid(method, config['enhancers'][method]):\n            f1s, timings = [], []\n            for condition in config['conditions']:\n                for noise in config['noise_seeds']:\n                    cached = cache_images(root, rows, manifest_hash, config, condition, noise, method, params, device)\n                    timings.append(np.mean([float(r['latency_ms']) for r in cached]))\n                    for model, saved in models:\n                        predictions = predict(root, rows, cached, model, config, device)\n                        score = classification([r['class_id'] for r in predictions], [r['prediction'] for r in predictions])\n                        f1s.append(score['macro_f1'])\n                        run = {'method': method, 'config_id': config_id(params), 'condition': condition,\n                               'noise_seed': noise, 'train_seed': saved['seed'], 'macro_f1': score['macro_f1'],\n                               'accuracy': score['accuracy'], 'weighted_f1': score['weighted_f1'], 'n_images': len(rows)}\n                        details.append(run)\n                        write_csv(output / 'predictions' / f'{config_id(run)}.csv', predictions)\n                    if release_cache:\n                        release_image_cache(root, cached)\n            candidates.append({'method': method, 'params': params, 'macro_f1': float(np.mean(f1s)),\n                               'latency_ms': float(np.mean(timings))})\n            print(f\"validation {method} {params}: {np.mean(f1s):.5f}\", flush=True)\n        winner = max(candidates, key=lambda r: (r['macro_f1'], -r['latency_ms']))\n        selected_params[method] = winner['params']\n        scores.append(winner)\n        write_csv(output / 'validation_runs.csv', details)\n        write_json(output / f'grid_{method}.json', candidates)\n    chosen = choose_pipeline(scores, config['selection_tolerance'])\n    result = {'status': 'completed', 'split': 'validation', 'manifest_sha256': manifest_hash,\n              'config': config, 'code_hash': source_code_hash(), 'environment': environment(),\n              'checkpoints': {str(Path(p).resolve()): digest(p) for p in checkpoints},\n              'sample_ids': [r['sample_id'] for r in rows], 'methods': list(methods),\n              'enhancer_provenance': {m: provenance(m, selected_params[m]) for m in methods},\n              'enhancers': selected_params, 'chosen_method': chosen['method'], 'scores': scores,\n              'validation_subset': per_class is not None,\n              'pilot': any(s['pilot'] for _, s in models) or set(methods) != set(METHODS),\n              'averaging': 'equal weight over conditions, noise seeds, then CNN seeds'}\n    write_json(output / 'selection.json', result)\n    return result\n\n\ndef freeze(root, selection_path, output, per_class=None, subset_seed=42, pilot=False):\n    output, root = Path(output), Path(root)\n    if output.exists():\n        raise FileExistsError('Protocol already frozen; use a new filename')\n    selection = json.loads(Path(selection_path).read_text())\n    rows, manifest_hash = manifest_rows(root)\n    config = deepcopy(selection['config'])\n    if selection['manifest_sha256'] != manifest_hash or selection['code_hash'] != source_code_hash():\n        raise ValueError('Selection is stale: manifest or implementation changed')\n    if selection['pilot'] and not pilot:\n        raise ValueError('Pilot validation needs --pilot; cannot label it final')\n    if set(selection['methods']) != set(METHODS) and not pilot:\n        raise ValueError('Final protocol requires identity and all seven enhancers')\n    checkpoints = []\n    seeds = []\n    for path, expected in selection['checkpoints'].items():\n        if digest(path) != expected:\n            raise ValueError('Checkpoint changed since validation')\n        saved = torch.load(path, map_location='cpu', weights_only=True)\n        seeds.append(saved['seed'])\n        checkpoints.append({'path': path, 'sha256': expected, 'seed': saved['seed']})\n    if not pilot and (set(seeds) != {11, 22, 33} or set(config['noise_seeds']) != {101, 202, 303}):\n        raise ValueError('Final protocol requires three training and three noise seeds')\n    for method, params in selection['enhancers'].items():\n        if provenance(method, params) != selection['enhancer_provenance'][method]:\n            raise ValueError(f'Enhancer changed since validation: {method}')\n        config['enhancers'][method] = params\n    if not pilot and set(config['conditions']) != {'mild', 'medium', 'severe'}:\n        raise ValueError('Final protocol requires all three degradation conditions')\n    test = subset_rows(rows, 'test', per_class, subset_seed)\n    runs = []\n    for checkpoint in checkpoints:\n        runs.append({'condition': 'clean', 'noise_seed': '', 'method': 'clean', 'train_seed': checkpoint['seed']})\n        for noise in config['noise_seeds']:\n            for condition in config['conditions']:\n                for method in selection['methods']:\n                    runs.append({'condition': condition, 'noise_seed': noise, 'method': method, 'train_seed': checkpoint['seed']})\n    snapshot = {'protocol': 'A', 'pilot': pilot, 'config': config, 'manifest_sha256': manifest_hash,\n                'selection_sha256': digest(selection_path), 'selection': selection,\n                'sample_ids': [r['sample_id'] for r in test], 'subset_seed': subset_seed,\n                'subset_id': 'official_test' if per_class is None else config_id([r['sample_id'] for r in test]),\n                'checkpoints': checkpoints, 'runs': runs, 'code_hash': source_code_hash(),\n                'enhancer_provenance': {m: provenance(m, config['enhancers'][m]) for m in selection['methods']},\n                'environment': environment()}\n    write_json(output, snapshot)\n    return snapshot\n\n\ndef evaluate(root, protocol_path, output, device, release_cache=False):\n    root, output = Path(root), Path(output)\n    protocol_path = Path(protocol_path)\n    protocol = json.loads(protocol_path.read_text())\n    config = protocol['config']\n    rows, manifest_hash = manifest_rows(root)\n    if manifest_hash != protocol['manifest_sha256'] or source_code_hash() != protocol['code_hash']:\n        raise ValueError('Frozen manifest or implementation changed')\n    by_id = {r['sample_id']: r for r in rows if r['split'] == 'test'}\n    if len(set(protocol['sample_ids'])) != len(protocol['sample_ids']) or not set(protocol['sample_ids']) <= by_id.keys():\n        raise ValueError('Invalid frozen test subset')\n    rows = [by_id[k] for k in protocol['sample_ids']]\n    models = {}\n    for c in protocol['checkpoints']:\n        if digest(c['path']) != c['sha256']:\n            raise ValueError('Frozen checkpoint changed')\n        models[c['seed']] = load_cnn(c['path'], manifest_hash, device, config['cnn'])[0]\n    for method, recorded in protocol['enhancer_provenance'].items():\n        if provenance(method, config['enhancers'][method]) != recorded:\n            raise ValueError(f'Frozen enhancer changed: {method}')\n    lock = output / 'protocol.json'\n    if lock.exists() and digest(lock) != digest(protocol_path):\n        raise ValueError('Output directory belongs to a different protocol')\n    if not lock.exists():\n        write_json(lock, protocol)\n    if not (output / 'classifier_latency.json').exists():\n        images = torch.stack([image_tensor(read_image(root / r['crop_path']))\n                              for r in rows[:config['cnn']['batch_size']]]).to(device)\n        write_json(output / 'classifier_latency.json', {\n            'cnn_config': cnn_config(config['cnn']),\n            'measurements': {str(seed): classifier_latency(model, images, device)\n                             for seed, model in models.items()}})\n    # Process all classifier seeds against the same cache before releasing it.\n    ordered_runs = protocol['runs']\n    if release_cache:\n        ordered_runs = sorted(ordered_runs, key=lambda r: (r['condition'], str(r['noise_seed']), r['method'], r['train_seed']))\n    states = [dict(run, status='planned') for run in ordered_runs]\n    summaries = []\n    for index, run in enumerate(ordered_runs):\n        run_id = config_id(run)\n        states[index]['status'] = 'running'\n        write_json(output / 'runs.json', states)\n        try:\n            params = config['enhancers'].get(run['method'], {})\n            cached = cache_images(root, rows, manifest_hash, config, run['condition'], run['noise_seed'], run['method'], params, device)\n            predictions = predict(root, rows, cached, models[run['train_seed']], config, device)\n            write_csv(output / 'predictions' / f'{run_id}.csv', predictions)\n            score = classification([r['class_id'] for r in predictions], [r['prediction'] for r in predictions])\n            write_json(output / 'classes' / f'{run_id}.json', score)\n            summaries.append({'dataset': 'gtsrb', 'split': 'test', 'protocol': 'A', 'subset_id': protocol['subset_id'],\n                              **run, 'run_id': run_id, 'config_id': config_id(params), 'n_images': len(rows),\n                              'accuracy': score['accuracy'], 'macro_f1': score['macro_f1'],\n                              'weighted_f1': score['weighted_f1'],\n                              'psnr_mean': float(np.mean([r['psnr'] for r in predictions])),\n                              'ssim_mean': float(np.mean([r['ssim'] for r in predictions])),\n                              'latency_ms': float(np.mean([r['latency_ms'] for r in predictions]))})\n            states[index]['status'] = 'completed'\n            write_csv(output / 'summary.csv', summaries)\n            key = lambda r: (r['condition'], str(r['noise_seed']), r['method'])\n            if release_cache and (index + 1 == len(ordered_runs) or key(ordered_runs[index + 1]) != key(run)):\n                release_image_cache(root, cached)\n            print(f\"{index+1}/{len(states)} {run}: macro_f1={score['macro_f1']:.4f}\", flush=True)\n        except Exception as error:\n            states[index].update(status='failed', error=str(error))\n            write_json(output / 'runs.json', states)\n            raise\n    write_json(output / 'runs.json', states)\n    write_json(output / 'environment.json', environment())\n    return summaries\n",
  "src/metrics.py": "from collections import defaultdict\n\nimport numpy as np\nfrom skimage.metrics import structural_similarity\nfrom .common import validate_image\n\n\ndef classification(labels, predictions):\n    labels, predictions = np.asarray(labels, dtype=int), np.asarray(predictions, dtype=int)\n    if labels.shape != predictions.shape or labels.ndim != 1 or labels.size == 0:\n        raise ValueError('Expected nonempty paired label/prediction vectors')\n    if min(labels.min(), predictions.min()) < 0 or max(labels.max(), predictions.max()) >= 43:\n        raise ValueError('Class IDs must be 0..42')\n    matrix = np.bincount(43 * labels + predictions, minlength=43*43).reshape(43, 43)\n    support, predicted = matrix.sum(1), matrix.sum(0)\n    f1 = np.divide(2*np.diag(matrix), support+predicted, out=np.zeros(43), where=support+predicted > 0)\n    return {'accuracy': float(np.trace(matrix) / labels.size), 'macro_f1': float(f1.mean()),\n            'weighted_f1': float(f1 @ support / labels.size), 'f1_per_class': f1.tolist(),\n            'support': support.tolist(), 'confusion_matrix': matrix.tolist()}\n\n\ndef image_metrics(reference, output, ssim_config):\n    reference, output = validate_image(reference), validate_image(output)\n    if reference.shape != output.shape:\n        raise ValueError('Reference/output shape mismatch')\n    mse = np.mean((reference.astype(np.float64)-output)**2)\n    return {'psnr': float('inf') if mse == 0 else float(10*np.log10(1/mse)),\n            'ssim': float(structural_similarity(reference, output, **ssim_config))}\n\n\ndef paired_rows(baseline, enhanced):\n    key = lambda r: (r['sample_id'], str(r['noise_seed']))\n    a, b = {key(r): r for r in baseline}, {key(r): r for r in enhanced}\n    if len(a) != len(baseline) or len(b) != len(enhanced) or a.keys() != b.keys():\n        raise ValueError('Paired sample IDs/noise seeds mismatch or duplicate')\n    pairs = [(a[k], b[k]) for k in sorted(a)]\n    if any(x['class_id'] != y['class_id'] or x['group_id'] != y['group_id'] for x, y in pairs):\n        raise ValueError('Paired labels/groups mismatch')\n    return pairs\n\n\ndef paired_bootstrap(baseline, enhanced, repetitions=1000, seed=42):\n    pairs = paired_rows(baseline, enhanced)\n    groups = defaultdict(list)\n    for i, (a, _) in enumerate(pairs):\n        groups[a['group_id']].append(i)\n    if repetitions < 1:\n        raise ValueError('Bootstrap repetitions must be positive')\n    values = list(groups.values())\n    labels = np.array([int(a['class_id']) for a, _ in pairs])\n    before = np.array([int(a['prediction']) for a, _ in pairs])\n    after = np.array([int(b['prediction']) for _, b in pairs])\n    noise = np.array([str(a['noise_seed']) for a, _ in pairs])\n    seeds = np.unique(noise)\n\n    def delta(indices):\n        # Same resampled source groups for every noise realization; average metrics over noise.\n        result = []\n        for ns in seeds:\n            idx = indices[noise[indices] == ns]\n            a, b = classification(labels[idx], before[idx]), classification(labels[idx], after[idx])\n            result.append([100*(b[k]-a[k]) for k in ['accuracy', 'macro_f1']])\n        return np.mean(result, axis=0)\n\n    rng = np.random.default_rng(seed)\n    draws = [delta(np.concatenate([values[i] for i in rng.integers(len(values), size=len(values))]))\n             for _ in range(repetitions)]\n    intervals = np.percentile(draws, [2.5, 97.5], axis=0)\n    observed = delta(np.arange(len(pairs)))\n    return {'n_groups': len(groups), 'repetitions': repetitions, 'unit': 'percentage points',\n            'conditional_on_checkpoint': True, 'grouping': 'group_id; test falls back to source_id',\n            **{k: {'delta': float(observed[i]), 'ci95': intervals[:, i].tolist()}\n               for i, k in enumerate(['accuracy', 'macro_f1'])}}\n",
  "src/model.py": "import math\n\nimport torch\nfrom torch import nn\n\n\ndef cnn_config(config=None):\n    \"\"\"Resolve legacy defaults and reject incompatible classifier settings.\"\"\"\n    cfg = {'architecture': 'cnn', 'input_size': 64, 'normalization': 'none',\n           'weights': None, 'optimizer': 'adam', 'weight_decay': 0.0,\n           'min_delta': 0.0, **(config or {})}\n    expected = {'cnn': (64, 'none', None), 'resnet18': (224, 'imagenet', 'IMAGENET1K_V1')}\n    if cfg['architecture'] not in expected:\n        raise ValueError(f\"Unknown architecture: {cfg['architecture']}\")\n    if tuple(cfg[k] for k in ['input_size', 'normalization', 'weights']) != expected[cfg['architecture']]:\n        raise ValueError('Invalid classifier input_size, normalization or weights')\n    if cfg['optimizer'] not in {'adam', 'adamw'} or cfg['weight_decay'] < 0:\n        raise ValueError('Invalid classifier optimizer or weight_decay')\n    if not math.isfinite(cfg['min_delta']) or cfg['min_delta'] < 0:\n        raise ValueError('Invalid classifier min_delta: expected a finite nonnegative value')\n    return cfg\n\n\nclass ResNet18(nn.Module):\n    def __init__(self, pretrained=False):\n        super().__init__()\n        from torchvision.models import resnet18, ResNet18_Weights\n        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)\n        self.backbone.fc = nn.Linear(512, 43)\n\n    def forward(self, images):\n        images = nn.functional.interpolate(images, size=(224, 224), mode='bilinear',\n                                          align_corners=False, antialias=True)\n        return self.backbone((images - self.mean) / self.std)\n\n\ndef make_cnn(config=None, pretrained=False):\n    cfg = cnn_config(config)\n    if cfg['architecture'] == 'resnet18':\n        return ResNet18(pretrained)\n    return nn.Sequential(\n        nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),\n        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),\n        nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),\n        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(128, 43))\n\n\ndef image_tensor(image):\n    return torch.from_numpy(image.transpose(2, 0, 1).copy())\n",
  "src/reporting.py": "\"\"\"Generate tables and figures from completed paired runs, never invented scores.\"\"\"\nfrom collections import defaultdict\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom PIL import Image, ImageDraw\nfrom scipy.stats import pearsonr, spearmanr\n\nfrom .common import config_id, read_csv, read_image, write_csv, write_json\nfrom .metrics import classification, paired_bootstrap, paired_rows\n\n\ndef report(output, root=Path('.')):\n    output, root = Path(output), Path(root)\n    protocol = json.loads((output / 'protocol.json').read_text())\n    states = json.loads((output / 'runs.json').read_text())\n    if len(states) != len(protocol['runs']) or any(r['status'] != 'completed' for r in states):\n        raise ValueError('Incomplete run matrix; finish failed/planned runs before reporting')\n    rows = read_csv(output / 'summary.csv')\n    if {r['run_id'] for r in rows} != {config_id(r) for r in protocol['runs']} or len(rows) != len(states):\n        raise ValueError('Summary does not match frozen run matrix')\n    figures = output / 'figures'\n    figures.mkdir(exist_ok=True)\n    expected_ids = set(protocol['sample_ids'])\n    predictions = {}\n    for row in rows:\n        preds = read_csv(output / 'predictions' / f\"{row['run_id']}.csv\")\n        if len(preds) != len(expected_ids) or {p['sample_id'] for p in preds} != expected_ids:\n            raise ValueError('Run sample coverage differs from frozen subset')\n        predictions[row['run_id']] = preds\n        if not row.get('weighted_f1'):\n            row['weighted_f1'] = classification([int(p['class_id']) for p in preds],\n                                                 [int(p['prediction']) for p in preds])['weighted_f1']\n    baselines = {(r['condition'], r['train_seed'], r['noise_seed']): r for r in rows if r['method'] == 'identity'}\n    deltas = []\n    for row in rows:\n        if row['method'] == 'clean':\n            continue\n        base = baselines[(row['condition'], row['train_seed'], row['noise_seed'])]\n        paired_rows(predictions[base['run_id']], predictions[row['run_id']])\n        deltas.append({**row, **{'delta_'+k+'_pp': 100*(float(row[k])-float(base[k])) for k in ['accuracy', 'macro_f1', 'weighted_f1']},\n                       **{'delta_'+k: float(row[k])-float(base[k]) for k in ['psnr_mean', 'ssim_mean']}})\n    write_csv(output / 'deltas.csv', deltas)\n    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'psnr_mean', 'ssim_mean', 'latency_ms']\n    grouped = defaultdict(list)\n    for row in rows:\n        grouped[(row['condition'], row['method'], row['train_seed'])].append(row)\n    per_training_seed = []\n    for (condition, method, seed), members in grouped.items():\n        per_training_seed.append({'condition': condition, 'method': method, 'train_seed': seed,\n                                  **{k: float(np.mean([float(r[k]) for r in members])) for k in metrics}})\n    write_csv(output / 'noise_averaged.csv', per_training_seed)\n    # Preserve all seed combinations in summary.csv; this second table describes noise sensitivity.\n    sensitivity = []\n    for (condition, method, seed), members in grouped.items():\n        sensitivity.append({'condition': condition, 'method': method, 'train_seed': seed,\n                            **{k+'_noise_std': float(np.std([float(r[k]) for r in members], ddof=1)) if len(members)>1 else 0.0\n                               for k in ['accuracy', 'macro_f1', 'weighted_f1']}})\n    write_csv(output / 'noise_sensitivity.csv', sensitivity)\n    aggregate = []\n    for condition, method in sorted({(r['condition'], r['method']) for r in rows}):\n        members = [r for r in per_training_seed if r['condition'] == condition and r['method'] == method]\n        entry = {'condition': condition, 'method': method, 'n_training_seeds': len(members)}\n        for key in metrics:\n            values = [r[key] for r in members]\n            entry[key+'_mean'] = float(np.mean(values))\n            entry[key+'_std'] = 0.0 if len(members) == 1 or all(np.isinf(v) for v in values) else float(np.std(values, ddof=1))\n        aggregate.append(entry)\n    write_csv(output / 'aggregate.csv', aggregate)\n    correlations = []\n    for condition in protocol['config']['conditions']:\n        points = [r for r in aggregate if r['condition'] == condition]\n        for quality in ['psnr_mean', 'ssim_mean']:\n            for recognition in ['accuracy', 'macro_f1', 'weighted_f1']:\n                x, y = np.array([r[quality+'_mean'] for r in points]), np.array([r[recognition+'_mean'] for r in points])\n                valid = np.isfinite(x) & np.isfinite(y)\n                x, y = x[valid], y[valid]\n                defined = len(x) >= 3 and np.ptp(x)>0 and np.ptp(y)>0\n                correlations.append({'condition': condition, 'quality': quality, 'recognition': recognition,\n                                     'n': len(x), 'pearson': float(pearsonr(x,y).statistic) if defined else '',\n                                     'spearman': float(spearmanr(x,y).statistic) if defined else '',\n                                     'status': 'exploratory; dependent methods' if defined else 'undefined: constant or too few points'})\n                fig, ax = plt.subplots(figsize=(7, 5))\n                ax.scatter(x, 100*y)\n                for row in points:\n                    if np.isfinite(row[quality+'_mean']):\n                        ax.annotate(row['method'], (row[quality+'_mean'], 100*row[recognition+'_mean']), fontsize=8)\n                ax.set(xlabel=quality, ylabel=recognition+' (%)', title=f'{condition}; method means, n={len(x)}')\n                fig.tight_layout()\n                fig.savefig(figures / f'scatter_{condition}_{quality}_{recognition}.png', dpi=140)\n                plt.close(fig)\n    write_csv(output / 'correlations.csv', correlations)\n    for metric in ['accuracy', 'macro_f1', 'weighted_f1']:\n        fig, ax = plt.subplots(figsize=(9, 5))\n        levels = list(protocol['config']['conditions'])\n        for method in protocol['selection']['methods']:\n            points = [next(r for r in aggregate if r['condition']==c and r['method']==method) for c in levels]\n            ax.errorbar(levels, [100*r[metric+'_mean'] for r in points],\n                        yerr=[100*r[metric+'_std'] for r in points], marker='o', label=method)\n        ax.set(ylabel=metric+' (%)', title='Noise averaged first; error bars = training seed SD')\n        ax.legend(fontsize=8, ncol=2)\n        fig.tight_layout()\n        fig.savefig(figures / f'{metric}_levels.png', dpi=140)\n        plt.close(fig)\n    bootstrap, failures = [], []\n    selected = protocol['selection']['chosen_method']\n    for condition in protocol['config']['conditions']:\n        for checkpoint in protocol['checkpoints']:\n            seed = str(checkpoint['seed'])\n            a, b = [], []\n            for row in rows:\n                if row['condition'] == condition and row['train_seed'] == seed:\n                    if row['method'] == 'identity':\n                        a.extend(predictions[row['run_id']])\n                    if row['method'] == selected:\n                        b.extend(predictions[row['run_id']])\n            bootstrap.append({'condition': condition, 'train_seed': seed, 'method': selected,\n                              **paired_bootstrap(a,b,protocol['config']['bootstrap_repetitions'])})\n            for before, after in paired_rows(a,b):\n                old = before['prediction'] == before['class_id']\n                new = after['prediction'] == after['class_id']\n                category = 'helped' if new and not old else 'harmed' if old and not new else 'still_wrong' if not new else 'still_correct'\n                failures.append({'condition': condition, 'train_seed': seed, 'category': category,\n                                 'sample_id': after['sample_id'], 'noise_seed': after['noise_seed'],\n                                 'class_id': after['class_id'], 'before': before['prediction'], 'after': after['prediction'],\n                                 'before_path': before['output_path'], 'after_path': after['output_path']})\n    write_json(output / 'bootstrap.json', bootstrap)\n    write_csv(output / 'cases.csv', failures)\n    # First stable sample ID per category; never cherry-pick visually attractive examples.\n    examples = []\n    for category in ['helped', 'harmed', 'still_wrong']:\n        members = sorted([r for r in failures if r['category']==category], key=lambda r: (r['sample_id'], r['condition'], r['noise_seed']))\n        examples.extend(members[:5])\n    write_json(output / 'example_ids.json', examples)\n    if examples:\n        sheet = Image.new('RGB', (400, 150*len(examples)), 'white')\n        draw = ImageDraw.Draw(sheet)\n        for i, case in enumerate(examples):\n            for j, key in enumerate(['before_path', 'after_path']):\n                image = read_image(root / case[key])\n                sheet.paste(Image.fromarray(np.rint(image*255).astype(np.uint8)).resize((112,112)), (j*140,i*150))\n            draw.text((0,i*150+113), f\"{case['category']}: {case['before']} -> {case['after']} (true {case['class_id']})\", fill='black')\n        sheet.save(figures / 'failure_examples.png')\n    distribution = root / 'data/manifests/class_distribution.csv'\n    if distribution.exists():\n        counts = read_csv(distribution)\n        fig, ax = plt.subplots(figsize=(12,4))\n        for split in ['train','validation','test']:\n            members = [r for r in counts if r['split']==split]\n            ax.plot([int(r['class_id']) for r in members], [int(r['count']) for r in members], label=split)\n        ax.set(xlabel='Class ID', ylabel='Images', title='GTSRB class distribution')\n        ax.legend(); fig.tight_layout(); fig.savefig(figures/'class_distribution.png', dpi=140); plt.close(fig)\n    for checkpoint in protocol['checkpoints']:\n        learning = Path(checkpoint['path']).parent / f\"learning_{checkpoint['seed']}.csv\"\n        if learning.exists():\n            history = read_csv(learning)\n            fig, axes = plt.subplots(1,3,figsize=(13,4))\n            for ax, metric in zip(axes, ['loss','accuracy','macro_f1']):\n                for split in ['train','validation']:\n                    members = [r for r in history if r['split']==split]\n                    ax.plot([int(r['epoch']) for r in members], [float(r[metric]) for r in members], label=split)\n                ax.set(xlabel='Epoch', ylabel=metric); ax.legend()\n            fig.tight_layout(); fig.savefig(figures/f\"learning_{checkpoint['seed']}.png\", dpi=140); plt.close(fig)\n    for row in rows:\n        if row['method'] in {'clean', selected} and row['noise_seed'] in {'', str(protocol['config']['noise_seeds'][0])}:\n            score = json.loads((output / 'classes' / f\"{row['run_id']}.json\").read_text())\n            fig, ax = plt.subplots(figsize=(8, 7))\n            ax.imshow(score['confusion_matrix'], cmap='Blues')\n            ax.set(xlabel='Predicted class', ylabel='True class', title=f\"{row['method']} {row['condition']} seed {row['train_seed']}\")\n            fig.tight_layout(); fig.savefig(figures / f\"confusion_{row['run_id']}.png\", dpi=130); plt.close(fig)\n    class_rows = []\n    for row in rows:\n        preds = predictions[row['run_id']]\n        score = classification([int(p['class_id']) for p in preds], [int(p['prediction']) for p in preds])\n        for label, (support, f1) in enumerate(zip(score['support'], score['f1_per_class'])):\n            class_rows.append({**{k: row[k] for k in ['run_id', 'condition', 'method', 'train_seed', 'noise_seed']},\n                               'class_id': label, 'support': support, 'f1': f1})\n    write_csv(output / 'per_class.csv', class_rows)\n    lines = ['# GTSRB: báo cáo kết quả thực chạy', '',\n             '**PILOT — không phải kết quả đánh giá cuối.**' if protocol['pilot'] else 'Giao thức A: CNN khóa trọng số.', '',\n             f\"Classifier: {protocol['config']['cnn'].get('architecture', 'cnn')}; normalization: {protocol['config']['cnn']['normalization']}.\",\n             f\"Số mẫu mỗi điều kiện: {len(expected_ids)}; subset: `{protocol['subset_id']}`.\",\n             f\"Pipeline chọn trước test bằng validation: **{selected}**.\", '',\n             'Noise seed được lấy trung bình trong mỗi training seed, sau đó báo mean ± sample SD qua training seed.', '',\n             '| Mức | Phương pháp | Accuracy (%) | Macro-F1 (%) | Weighted-F1 (%) | PSNR (dB) | SSIM | Enhancer ms/ảnh |',\n             '|---|---|---:|---:|---:|---:|---:|---:|']\n    for r in aggregate:\n        lines.append(f\"| {r['condition']} | {r['method']} | {100*r['accuracy_mean']:.2f} ± {100*r['accuracy_std']:.2f} | {100*r['macro_f1_mean']:.2f} ± {100*r['macro_f1_std']:.2f} | {100*r['weighted_f1_mean']:.2f} ± {100*r['weighted_f1_std']:.2f} | {r['psnr_mean_mean']:.2f} | {r['ssim_mean_mean']:.4f} | {r['latency_ms_mean']:.2f} |\")\n    lines += ['', '## Phân tích và giới hạn', '',\n              '- `classifier_latency.json`: thời gian preprocessing + classifier riêng, có warm-up và đồng bộ; `latency_ms` trong bảng chỉ là enhancer.',\n              '- `deltas.csv`: chênh lệch so identity cùng mức và seed; nhận diện tính bằng điểm phần trăm.',\n              '- `correlations.csv`: Pearson/Spearman theo mức; loại clean; ít điểm và các phương pháp phụ thuộc nhau.',\n              '- `bootstrap.json`: paired bootstrap theo group/source, có điều kiện trên checkpoint.',\n              '- `per_class.csv`: support và F1 đủ 43 lớp, gồm class 19; confusion matrix trong `classes/` và `figures/`.',\n              '- `cases.csv`, `example_ids.json`: mẫu được giúp, làm hỏng, hoặc vẫn nhận sai; ID lựa chọn cố định.',\n              '- Đây là thiếu sáng mô phỏng trên crop GTSRB; không suy thành hiệu năng camera ban đêm.',\n              '- PSNR clean là vô hạn; không thay bằng số hữu hạn. Một training seed không ước lượng được biến thiên huấn luyện.', '',\n              '![Macro-F1](figures/macro_f1_levels.png)', '', '![Accuracy](figures/accuracy_levels.png)']\n    (output / 'report.md').write_text('\\n'.join(lines)+'\\n')\n    return output / 'report.md'\n",
  "src/training.py": "import json\nfrom pathlib import Path\nimport random\nimport time\nimport sys\ntry:\n    import resource\nexcept ImportError:  # Windows has no stdlib process RSS counter.\n    resource = None\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nimport torchvision.transforms as T\n\nfrom .common import digest, environment, read_csv, read_image, synchronize, write_csv, write_json\nfrom .data import subset_rows\nfrom .metrics import classification\nfrom .model import make_cnn, image_tensor, cnn_config\n\n\nclass CropDataset(Dataset):\n    def __init__(self, root, rows, transform=None):\n        self.root, self.rows = Path(root), rows\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        tensor = image_tensor(read_image(self.root / row['crop_path']))\n        if self.transform is not None:\n            tensor = self.transform(tensor)\n        return tensor, int(row['class_id'])\n\n\nTRAIN_TRANSFORM = T.Compose([\n    T.RandomRotation(degrees=10),\n    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),\n    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),\n    T.RandomErasing(p=0.1, scale=(0.02, 0.08)),\n])\n\n\ndef train(root, config, seed, device, output, per_class=None, epochs=None):\n    root, output = Path(root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    checkpoint = output / f'cnn_{seed}.pt'\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n    cfg = cnn_config(config['cnn'])\n    count_epochs = cfg['epochs'] if epochs is None else epochs\n    if count_epochs < 1 or cfg['batch_size'] < 1 or cfg['patience'] < 1 or cfg['learning_rate'] <= 0:\n        raise ValueError('Invalid training configuration')\n    manifest = root / 'data/manifests/gtsrb_samples.csv'\n    meta = json.loads(manifest.with_suffix('.json').read_text())\n    if digest(manifest) != meta['sha256']:\n        raise ValueError('Manifest has changed')\n    audit_path = manifest.parent / 'audit.json'\n    if not audit_path.exists() or json.loads(audit_path.read_text())['status'] != 'completed':\n        raise ValueError('Data audit is missing or failed; run check-data')\n    rows = read_csv(manifest)\n    train_rows = subset_rows(rows, 'train', per_class)\n    validation_rows = subset_rows(rows, 'validation', per_class)\n    metadata = {'seed': seed, 'config': config, 'manifest_sha256': digest(manifest),\n                'environment': environment(), 'device': str(device),\n                'requested_epochs': count_epochs,\n                'pilot': per_class is not None or count_epochs != cfg['epochs'],\n                'train_ids': [r['sample_id'] for r in train_rows],\n                'validation_ids': [r['sample_id'] for r in validation_rows]}\n    if checkpoint.exists():\n        saved = torch.load(checkpoint, map_location='cpu', weights_only=True)\n        log = output / f'train_{seed}.json'\n        recorded = json.loads(log.read_text()) if log.exists() else {}\n        matches = (cnn_config(saved.get('cnn_config')) == cfg\n                   and saved.get('manifest_sha256') == metadata['manifest_sha256']\n                   and saved.get('seed') == seed and recorded.get('status') == 'completed'\n                   and recorded.get('checkpoint_sha256') == digest(checkpoint)\n                   and all(recorded.get(k) == metadata[k] for k in\n                           ['config', 'requested_epochs', 'pilot', 'train_ids', 'validation_ids']))\n        if not matches:\n            raise ValueError('Existing checkpoint metadata/config/manifest mismatch or incomplete training; use a new --output')\n        print(f'{checkpoint} verified; skipping seed {seed}', flush=True)\n        return checkpoint\n    loaders = {s: DataLoader(CropDataset(root, r, transform=TRAIN_TRANSFORM if s == 'train' else None),\n                            batch_size=cfg['batch_size'], shuffle=s == 'train',\n                            num_workers=0, generator=torch.Generator().manual_seed(seed))\n               for s, r in [('train', train_rows), ('validation', validation_rows)]}\n    model = make_cnn(cfg, pretrained=True).to(device)\n    optimizer_class = torch.optim.AdamW if cfg['optimizer'] == 'adamw' else torch.optim.Adam\n    optimizer = optimizer_class(model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])\n    history, best, stopping_best, stale = [], -1, -1, 0\n    write_json(output / f'train_{seed}.json', dict(metadata, status='running'))\n    for epoch in range(1, count_epochs+1):\n        synchronize(device)\n        epoch_start = time.perf_counter()\n        if device.type == 'cuda':\n            torch.cuda.reset_peak_memory_stats(device)\n        for split, loader in loaders.items():\n            model.train(split == 'train')\n            labels, predictions, total_loss = [], [], 0.0\n            with torch.set_grad_enabled(split == 'train'):\n                for images, targets in loader:\n                    images, targets = images.to(device), targets.to(device)\n                    logits = model(images)\n                    loss = torch.nn.functional.cross_entropy(logits, targets)\n                    if not torch.isfinite(loss):\n                        raise ValueError('Nonfinite CNN loss')\n                    if split == 'train':\n                        optimizer.zero_grad(set_to_none=True)\n                        loss.backward()\n                        optimizer.step()\n                    labels.extend(targets.cpu().tolist())\n                    predictions.extend(logits.detach().argmax(1).cpu().tolist())\n                    total_loss += float(loss.detach()) * len(targets)\n            score = classification(labels, predictions)\n            history.append({'epoch': epoch, 'split': split, 'loss': total_loss/len(labels),\n                            'accuracy': score['accuracy'], 'macro_f1': score['macro_f1'],\n                            'weighted_f1': score['weighted_f1']})\n            if split == 'validation':\n                print(f\"seed={seed} epoch={epoch} val_loss={total_loss/len(labels):.4f} macro_f1={score['macro_f1']:.4f}\", flush=True)\n                if score['macro_f1'] > stopping_best + cfg['min_delta']:\n                    stopping_best, stale = score['macro_f1'], 0\n                else:\n                    stale += 1\n                if score['macro_f1'] > best:\n                    best = score['macro_f1']\n                    temporary = checkpoint.with_suffix('.tmp')\n                    torch.save({'state_dict': model.state_dict(), 'seed': seed, 'epoch': epoch,\n                                'macro_f1': best, 'manifest_sha256': digest(manifest),\n                                'cnn_config': cfg, 'pilot': metadata['pilot'],\n                                'environment': metadata['environment']}, temporary)\n                    temporary.replace(checkpoint)\n                    write_json(output / f'validation_{seed}.json', score)\n        synchronize(device)\n        for entry in history[-2:]:\n            entry['epoch_seconds'] = time.perf_counter() - epoch_start\n            entry['peak_rss_bytes'] = (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n                                       * (1 if sys.platform == 'darwin' else 1024)) if resource else None\n            entry['accelerator_bytes'] = (torch.cuda.max_memory_allocated(device) if device.type == 'cuda'\n                                          else torch.mps.driver_allocated_memory() if device.type == 'mps' else 0)\n        write_csv(output / f'learning_{seed}.csv', history)\n        if stale >= cfg['patience']:\n            print(f\"seed={seed} early stop at epoch={epoch}: patience={stale}, min_delta={cfg['min_delta']}\", flush=True)\n            break\n    write_json(output / f'train_{seed}.json', dict(metadata, status='completed',\n               checkpoint_sha256=digest(checkpoint), best_macro_f1=best))\n    return checkpoint\n",
  "src/tools/__init__.py": "\"\"\"Tracked entry points for reproducing and auditing the coursework.\"\"\"\n",
  "src/tools/public_sources.py": "\"\"\"Fetch upstream code/checkpoints directly; never upload or relicense those assets.\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport urllib.request\n\n\ndef provision(root, specification):\n    from src.common import digest\n    root = Path(root).resolve()\n    spec = json.loads(Path(specification).read_text(encoding='utf-8'))\n    records = []\n    for item in spec['models']:\n        target = root / item['directory']\n        if not target.exists():\n            subprocess.run(['git','clone','--quiet','--depth','1',item['repository'],str(target)],check=True)\n        actual = subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip()\n        if actual != item['commit']:\n            subprocess.run(['git','-C',str(target),'fetch','--quiet','--depth','1','origin',item['commit']],check=True)\n            subprocess.run(['git','-C',str(target),'checkout','--quiet','--detach',item['commit']],check=True)\n        checkpoint = root / item['checkpoint']\n        if not checkpoint.exists():\n            checkpoint.parent.mkdir(parents=True,exist_ok=True)\n            if item['download']['kind'] == 'gdrive':\n                import gdown\n                result = gdown.download(id=item['download']['id'],output=str(checkpoint),quiet=True)\n                if not result:\n                    raise RuntimeError(f\"Author checkpoint unavailable: {item['name']}\")\n            else:\n                with urllib.request.urlopen(item['download']['url'],timeout=120) as response, checkpoint.open('wb') as stream:\n                    import shutil\n                    shutil.copyfileobj(response,stream)\n        sha = digest(checkpoint)\n        if item.get('sha256') and sha != item['sha256']:\n            raise ValueError(f\"Checkpoint checksum mismatch: {item['name']}\")\n        records.append({'method':item['name'],'repository':item['repository'],'commit':item['commit'],\n                        'checkpoint_sha256':sha,'source':item['source_page'],'license_note':item['license_note']})\n        print('Verified upstream asset:',item['name'],sha,flush=True)\n    return records\n\n\nif __name__ == '__main__':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--root',type=Path,default=Path('.'))\n    p.add_argument('--spec',type=Path,default=Path('configs/public_sources.json'))\n    a=p.parse_args()\n    print(json.dumps(provision(a.root,a.spec),indent=2))\n",
  "src/tools/fresh_run.py": "\"\"\"Fresh Kaggle execution with official inputs, bounded caches and separate GPU workers.\"\"\"\nfrom collections import defaultdict\nfrom copy import deepcopy\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\ndef parallel_evaluate(root, protocol_path, output, devices):\n    from src.common import config_id, read_csv, write_csv, write_json\n    from .research_evidence import audited_runs\n    root,output=Path(root),Path(output)\n    master=json.loads(Path(protocol_path).read_text())\n    output.mkdir(parents=True,exist_ok=True)\n    lock=output/'protocol.json'\n    if lock.exists() and json.loads(lock.read_text()) != master:\n        raise ValueError('Output belongs to another experiment')\n    write_json(lock,master)\n    groups=defaultdict(list)\n    for r in master['runs']:groups[r['condition'],str(r['noise_seed']),r['method']].append(r)\n    assignments=[[] for _ in devices]\n    loads=[0 for _ in devices]\n    costs={'diffusion':30,'retinexformer':4,'zero_dce':2}\n    for key,runs in sorted(groups.items(),key=lambda item:(-costs.get(item[0][2],1),item[0])):\n        rank=min(range(len(devices)),key=lambda i:loads[i])\n        assignments[rank].extend(runs);loads[rank]+=costs.get(key[2],1)\n    workers=[]\n    for rank,(device,runs) in enumerate(zip(devices,assignments)):\n        if not runs:continue\n        partial=deepcopy(master);partial['runs']=runs;partial['pilot']=True\n        protocol=output/'_workers'/f'worker_{rank}.json'\n        write_json(protocol,partial)\n        dest=output/'_workers'/f'worker_{rank}'\n        command=[sys.executable,'-m','src.tools.fresh_run','evaluate-worker','--root',str(root),\n                 '--protocol',str(protocol),'--output',str(dest),'--device',device]\n        workers.append((dest,subprocess.Popen(command,cwd=root)))\n    codes=[process.wait() for _,process in workers]\n    if any(codes):raise RuntimeError(f'Evaluation worker exit codes: {codes}')\n    summaries=[]\n    for folder,_ in workers:\n        audited_runs(folder)\n        summaries.extend(read_csv(folder/'summary.csv'))\n        for name in ['predictions','classes']:\n            (output/name).mkdir(exist_ok=True)\n            for p in (folder/name).glob('*'):shutil.copy2(p,output/name/p.name)\n    by_id={r['run_id']:r for r in summaries}\n    ordered=[by_id[config_id(r)] for r in master['runs']]\n    write_csv(output/'summary.csv',ordered)\n    write_json(output/'runs.json',[dict(r,status='completed') for r in master['runs']])\n    write_json(output/'worker_devices.json',{'devices':devices,'grouping':'condition/noise/method; all CNN checkpoints share one generated cache',\n                                           'cache_policy':'release after every complete group'})\n    audited_runs(output)\n    return output\n\n\ndef ablation(root, config, selection, checkpoints, device, output):\n    from src.common import digest, read_csv, write_csv, write_json, config_id\n    from src.data import subset_rows\n    from src.experiment import cache_images, predict, load_cnn, release_image_cache\n    from src.enhancement import METHODS\n    from src.metrics import classification\n    root,output=Path(root),Path(output)\n    manifest=root/'data/manifests/gtsrb_samples.csv'\n    rows=subset_rows(read_csv(manifest),'test',10,42)\n    models=[load_cnn(p,digest(manifest),device,config['cnn']) for p in checkpoints]\n    base=config['conditions']['medium']\n    variants={'gamma_only':{'gamma_dark':base['gamma_dark']},\n              'gamma_gaussian':{'gamma_dark':base['gamma_dark'],'gaussian_sigma':base['gaussian_sigma']},\n              'gamma_poisson':{'gamma_dark':base['gamma_dark'],'poisson_peak':base['poisson_peak']},\n              'gamma_both':base}\n    write_json(output/'protocol.json',{'role':'supplemental ablation; no training/tuning','variants':variants,\n               'sample_ids':[r['sample_id'] for r in rows],'checkpoint_hashes':{p.name:digest(p) for p in checkpoints}})\n    results=[]\n    for variant,parameters in variants.items():\n        cfg=deepcopy(config);cfg['conditions']={'medium':parameters};cfg['enhancers'].update(selection['enhancers'])\n        for noise in cfg['noise_seeds']:\n            for method in METHODS:\n                cached=cache_images(root,rows,digest(manifest),cfg,'medium',noise,method,cfg['enhancers'][method],device)\n                for model,saved in models:\n                    predictions=predict(root,rows,cached,model,cfg,device)\n                    score=classification([r['class_id'] for r in predictions],[r['prediction'] for r in predictions])\n                    run={'variant':variant,'method':method,'noise_seed':noise,'train_seed':saved['seed']}\n                    rid=config_id(run)\n                    write_csv(output/'predictions'/f'{rid}.csv',predictions)\n                    results.append({**run,'run_id':rid,'n_images':len(rows),**{k:score[k] for k in ['accuracy','macro_f1','weighted_f1']}})\n                write_csv(output/'summary.csv',results)\n                release_image_cache(root,cached)\n                print(f'Ablation {len(results)}/288: {variant} {method}',flush=True)\n    if len(results)!=288:raise ValueError('Incomplete ablation')\n    return output\n\n\ndef train_fresh(root, config, output, devices):\n    from concurrent.futures import ThreadPoolExecutor\n    from src.training import train\n    # Independent seeds can train simultaneously; no DDP, no change to each seed's batch/optimizer.\n    def one(index,seed):\n        command=[sys.executable,'-m','src.tools.fresh_run','train-worker','--root',str(root),\n                 '--output',str(output),'--device',devices[index%len(devices)],'--seed',str(seed)]\n        subprocess.run(command,cwd=root,check=True)\n    # Assign each device its sequential seed queue to prevent two jobs sharing one GPU.\n    def queue(rank):\n        for index,seed in enumerate(config['train_seeds']):\n            if index%len(devices)==rank:one(index,seed)\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\n        list(pool.map(queue,range(len(devices))))\n    return [Path(output)/f'cnn_{s}.pt' for s in config['train_seeds']]\n\n\nif __name__=='__main__':\n    import argparse\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument('task',choices=['train-worker','evaluate-worker'])\n    p.add_argument('--root',type=Path,required=True)\n    p.add_argument('--output',type=Path,required=True)\n    p.add_argument('--device',required=True)\n    p.add_argument('--seed',type=int)\n    p.add_argument('--protocol',type=Path)\n    a=p.parse_args()\n    from src.common import device_for\n    import torch\n    torch.set_num_threads(2)\n    if a.task=='train-worker':\n        from src.training import train\n        cfg=json.loads((a.root/'configs/resnet18.json').read_text())\n        train(a.root,cfg,a.seed,device_for(a.device),a.output)\n    else:\n        from src.experiment import evaluate\n        evaluate(a.root,a.protocol,a.output,device_for(a.device),release_cache=True)\n",
  "src/tools/research_evidence.py": "\"\"\"Audit raw prediction coverage before producing error analysis or an evidence ZIP.\"\"\"\nimport csv\nimport hashlib\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\n\nfrom src.common import config_id, digest\nfrom src.metrics import classification, paired_rows\n\n\ndef read_csv(path):\n    with Path(path).open(encoding='utf-8-sig', newline='') as stream:\n        return list(csv.DictReader(stream))\n\n\ndef write_csv(path, rows, fields=None):\n    rows = list(rows)\n    with Path(path).open('w', encoding='utf-8', newline='') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields or list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef audited_runs(folder):\n    folder = Path(folder)\n    protocol = json.loads((folder / 'protocol.json').read_text(encoding='utf-8'))\n    states = json.loads((folder / 'runs.json').read_text(encoding='utf-8'))\n    expected = {config_id(r): r for r in protocol['runs']}\n    if len(expected) != len(protocol['runs']):\n        raise ValueError('Duplicate frozen runs')\n    if len(states) != len(expected) or {config_id({k: s[k] for k in ['condition', 'method', 'noise_seed', 'train_seed']}) for s in states} != set(expected) or any(s['status'] != 'completed' for s in states):\n        raise ValueError('Run states are incomplete or differ from the frozen protocol')\n    summaries = read_csv(folder / 'summary.csv')\n    if len(summaries) != len(expected) or {r['run_id'] for r in summaries} != set(expected):\n        raise ValueError('Summary does not cover the frozen run matrix')\n    ids = set(protocol['sample_ids'])\n    if not ids or len(ids) != len(protocol['sample_ids']):\n        raise ValueError('Invalid sample IDs in protocol')\n    reference_labels = None\n    loaded = []\n    for summary in summaries:\n        run = expected[summary['run_id']]\n        if any(str(summary[k]) != str(v) for k, v in run.items()):\n            raise ValueError('Summary metadata does not match frozen run')\n        if int(summary['n_images']) != len(ids):\n            raise ValueError('Incorrect sample count')\n        rows = read_csv(folder / 'predictions' / f\"{summary['run_id']}.csv\")\n        if len(rows) != len(ids) or {r['sample_id'] for r in rows} != ids:\n            raise ValueError('Missing or duplicate predictions')\n        for r in rows:\n            r['class_id'], r['prediction'] = int(r['class_id']), int(r['prediction'])\n            if r['noise_seed'] != str(run['noise_seed']):\n                raise ValueError('Prediction noise seed mismatch')\n        labels = {r['sample_id']: (r['class_id'], r['group_id']) for r in rows}\n        if reference_labels is None:\n            reference_labels = labels\n        elif labels != reference_labels:\n            raise ValueError('Label/group mismatch between runs')\n        score = classification([r['class_id'] for r in rows], [r['prediction'] for r in rows])\n        for metric in ['accuracy', 'macro_f1', 'weighted_f1']:\n            if not math.isclose(score[metric], float(summary[metric]), rel_tol=0, abs_tol=1e-8):\n                raise ValueError(f'Metric mismatch: {summary[\"run_id\"]} {metric}')\n        loaded.append((summary, rows, score))\n    if not protocol.get('pilot', True):\n        from src.enhancement import METHODS\n        methods=METHODS\n        if protocol.get('study')=='denoise_order_extension':\n            methods=protocol['selection']['methods']\n            if (len(methods)!=len(set(methods)) or set(methods)!=set(protocol['registry'])\n                or not {'identity','gamma','clahe'}<=set(methods)\n                or protocol['selection']['chosen_method'] not in methods):\n                raise ValueError('Invalid extension method registry or controls')\n            if any(protocol['registry'][m]!=protocol['selection']['registry'][m] for m in methods):\n                raise ValueError('Extension pipeline differs from validation selection')\n            intended=(protocol['selection']['fulltest_methods'] if protocol['subset_id']=='official_test'\n                      else list(protocol['selection']['registry']))\n            if set(methods)!=set(intended):raise ValueError('Extension test methods differ from selection')\n        elif protocol.get('study') is not None:\n            raise ValueError('Unknown study')\n        cfg = protocol['config']\n        if set(cfg['conditions']) != {'mild', 'medium', 'severe'} or set(cfg['noise_seeds']) != {101, 202, 303} or {c['seed'] for c in protocol['checkpoints']} != {11, 22, 33}:\n            raise ValueError('Final experiment requires all three levels and seeds')\n        required = {\n            config_id(dict(condition=c, method=m, noise_seed=n, train_seed=t))\n            for c in cfg['conditions'] for m in methods for n in cfg['noise_seeds'] for t in [11, 22, 33]\n        } | {config_id(dict(condition='clean', method='clean', noise_seed='', train_seed=t)) for t in [11, 22, 33]}\n        if set(expected) != required or {r['class_id'] for r in loaded[0][1]} != set(range(43)):\n            raise ValueError('Final experiment matrix/classes incomplete')\n        if protocol['subset_id'] == 'official_test' and len(ids) != 12630:\n            raise ValueError('Official full-test must contain 12,630 samples')\n    return protocol, loaded\n\n\ndef analyze(folder, output):\n    protocol, runs = audited_runs(folder)\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError('Use a new error-analysis output directory')\n    output.mkdir(parents=True)\n    baselines = {(r['condition'], r['train_seed'], r['noise_seed']): p for r,p,_ in runs if r['method'] == 'identity'}\n    per_class, transitions, confusions, cases = [], [], [], []\n    for run, predictions, score in runs:\n        meta = {k: run[k] for k in ['run_id', 'condition', 'method', 'train_seed', 'noise_seed']}\n        matrix = np.array(score['confusion_matrix'])\n        np.savetxt(output / f\"confusion_{run['run_id']}.csv\", matrix, delimiter=',', fmt='%d')\n        normalized = np.divide(matrix, matrix.sum(1, keepdims=True), out=np.zeros_like(matrix, dtype=float), where=matrix.sum(1, keepdims=True) != 0)\n        np.savetxt(output / f\"confusion_normalized_{run['run_id']}.csv\", normalized, delimiter=',', fmt='%.8f')\n        for c in range(43):\n            per_class.append({**meta, 'class_id': c, 'support': score['support'][c], 'f1': score['f1_per_class'][c]})\n            for p in range(43):\n                if p != c and matrix[c,p]:\n                    confusions.append({**meta, 'true_class': c, 'predicted_class': p, 'count': int(matrix[c,p]), 'row_fraction': float(normalized[c,p])})\n        if run['method'] == 'clean':\n            continue\n        key = (run['condition'], run['train_seed'], run['noise_seed'])\n        if key not in baselines:\n            raise ValueError('Missing matched identity baseline')\n        counts = Counter()\n        for before, after in paired_rows(baselines[key], predictions):\n            old = before['prediction'] == before['class_id']\n            new = after['prediction'] == after['class_id']\n            category = 'helped' if new and not old else 'harmed' if old and not new else 'still_correct' if new else 'still_wrong'\n            counts[category] += 1\n            cases.append({**meta, 'sample_id': after['sample_id'], 'class_id': after['class_id'], 'category': category,\n                          'before': before['prediction'], 'after': after['prediction'],\n                          'before_path': before['output_path'], 'after_path': after['output_path']})\n        transitions.append({**meta, **{k: counts[k] for k in ['helped', 'harmed', 'still_correct', 'still_wrong']},\n                            'net_correct': counts['helped']-counts['harmed']})\n    write_csv(output / 'per_class_runs.csv', per_class)\n    write_csv(output / 'transitions.csv', transitions)\n    write_csv(output / 'cases.csv', cases)\n    write_csv(output / 'confusion_pairs.csv', confusions, ['run_id','condition','method','train_seed','noise_seed','true_class','predicted_class','count','row_fraction'])\n    # Average F1 within checkpoint over noise, then across checkpoints. Never pool seeds.\n    groups = defaultdict(list)\n    for r in per_class:\n        groups[r['condition'],r['method'],r['class_id'],r['train_seed']].append(r['f1'])\n    across_seeds = defaultdict(list)\n    for (c,m,label,_), values in groups.items():\n        across_seeds[c,m,label].append(float(np.mean(values)))\n    baseline = {(c,label): np.mean(v) for (c,m,label),v in across_seeds.items() if m == 'identity'}\n    aggregates = []\n    for (c,m,label),values in sorted(across_seeds.items()):\n        mean = float(np.mean(values))\n        aggregates.append(dict(condition=c,method=m,class_id=label,f1_mean=mean,\n                               f1_sd=float(np.std(values, ddof=1)) if len(values)>1 else '',\n                               n_train_seeds=len(values), delta_f1_pp=100*(mean-baseline[c,label]) if (c,label) in baseline else ''))\n    write_csv(output / 'per_class_aggregate.csv', aggregates)\n    # IDs, method and seeds are retained; repeated source images are not independent examples.\n    examples = []\n    selected = protocol['selection']['chosen_method']\n    for method in sorted({selected, 'clahe'}):\n        for category in ['helped','harmed','still_wrong']:\n            subset = sorted((r for r in cases if r['method']==method and r['category']==category), key=lambda r: (r['sample_id'],r['condition'],r['train_seed'],r['noise_seed']))\n            seen = set()\n            for r in subset:\n                if r['sample_id'] not in seen:\n                    examples.append(r)\n                    seen.add(r['sample_id'])\n                    if len(seen) == 5:\n                        break\n    write_csv(output / 'examples.csv', examples, list(cases[0]))\n    import matplotlib\n    matplotlib.use('Agg')\n    import matplotlib.pyplot as plt\n    for run, _, score in runs:\n        if run['method'] not in {'identity', selected, 'clahe'} or str(run['train_seed']) != str(protocol['checkpoints'][0]['seed']) or str(run['noise_seed']) != str(protocol['config']['noise_seeds'][0]):\n            continue\n        matrix = np.array(score['confusion_matrix'], dtype=float)\n        matrix = np.divide(matrix, matrix.sum(1, keepdims=True), out=np.zeros_like(matrix), where=matrix.sum(1, keepdims=True)!=0)\n        fig, ax = plt.subplots(figsize=(9,8))\n        im = ax.imshow(matrix, vmin=0, vmax=1, cmap='Blues')\n        ax.set(xlabel='Predicted class', ylabel='True class', title=f\"{run['method']} / {run['condition']} / train {run['train_seed']} / noise {run['noise_seed']}\")\n        ax.set_xticks(range(0,43,2)); ax.set_yticks(range(0,43,2))\n        fig.colorbar(im, ax=ax, label='Fraction within true class')\n        fig.tight_layout(); fig.savefig(output / f\"confusion_{run['run_id']}.png\", dpi=150); plt.close(fig)\n    audit = {'status':'raw_predictions_recomputed', 'runs':len(runs), 'images_per_run':len(protocol['sample_ids']),\n             'scope':protocol['subset_id'], 'pilot':protocol.get('pilot',True),\n             'protocol_sha256':digest(Path(folder)/'protocol.json'),\n             'source_hashes':{str(p.relative_to(folder)):digest(p) for p in sorted(Path(folder).rglob('*.csv')) if p.parent.name=='predictions' or p.name=='summary.csv'},\n             'aggregation':'Average noise within checkpoint, then mean and sample SD across checkpoints; confusion counts remain per run'}\n    (output / 'audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')\n    return output\n\n\ndef bundle(folder, output):\n    audited_runs(folder)\n    folder, output = Path(folder), Path(output)\n    if output.exists():\n        raise FileExistsError('Use a new ZIP path')\n    required = ['protocol.json','runs.json','summary.csv']\n    paths = [folder / p for p in required]\n    paths += sorted((folder/'predictions').glob('*.csv'))\n    for name in ['classes','train','validation','error_analysis','error_analysis_prod']:\n        paths += sorted(p for p in (folder/name).rglob('*') if p.is_file())\n    for name in ['environment.json','classifier_latency.json','aggregate.csv','per_class.csv','bootstrap.json','cases.csv']:\n        if (folder/name).is_file():\n            paths.append(folder/name)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    manifest = {str(p.relative_to(folder)).replace('\\\\','/'):digest(p) for p in paths}\n    with zipfile.ZipFile(output,'x',compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in paths:\n            archive.write(p,p.relative_to(folder).as_posix())\n        archive.writestr('SHA256.json',json.dumps(manifest,indent=2))\n        archive.writestr('SCOPE.txt','Raw prediction evidence. Does not include original images, cached enhancer arrays, external weights or classifier checkpoints. Preserve those separately.\\n')\n    return output\n",
  "src/tools/result_archive.py": "\"\"\"A reader-friendly ZIP: merged prediction tables, explicit scope, hashes, no input images.\"\"\"\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\n\nfrom .research_evidence import audited_runs, read_csv\n\n\ndef aggregate(summaries):\n    metrics=['accuracy','macro_f1','weighted_f1','psnr_mean','ssim_mean','latency_ms']\n    within=defaultdict(list)\n    for r in summaries:\n        within[r['condition'],r['method'],r['train_seed']].append(r)\n    between=defaultdict(list)\n    for (condition,method,_),rows in within.items():\n        between[condition,method].append({k:float(np.mean([float(r[k]) for r in rows])) for k in metrics})\n    result=[]\n    for (condition,method),rows in sorted(between.items()):\n        out={'condition':condition,'method':method,'n_training_seeds':len(rows)}\n        for k in metrics:\n            values=[r[k] for r in rows]\n            out[k+'_mean']=float(np.mean(values))\n            out[k+'_sd']=0.0 if len(values)==1 or all(np.isinf(values)) else float(np.std(values,ddof=1))\n        result.append(out)\n    return result\n\n\ndef pack(folder, output, analysis=None, source=None, notebook=None, supplemental=None, include_checkpoints=True, subset=None, extension=None):\n    folder,output=Path(folder),Path(output)\n    protocol,runs=audited_runs(folder)\n    if output.exists():\n        raise FileExistsError('Use a new archive path')\n    output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=output.with_suffix('.partial.zip')\n    if temporary.exists():\n        temporary.unlink()\n    metadata={'schema':'gtsrb-readable-results-v1','status':'completed_and_prediction_audited',\n              'scope':protocol['subset_id'],'images_per_run':len(protocol['sample_ids']),\n              'runs':len(runs),'prediction_rows':sum(len(p) for _,p,_ in runs),\n              'aggregation':'Average noise within checkpoint, then mean and sample SD across training checkpoints.',\n              'not_included':['GTSRB source images','author model weights','image caches','duplicate demo image galleries'],\n              'cache_paths':'Any output_path column refers to regenerable temporary cache, not a file in this ZIP.'}\n    def csv_bytes(rows):\n        rows=list(rows)\n        stream=io.StringIO(newline='')\n        w=csv.DictWriter(stream,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n        return stream.getvalue().encode('utf-8')\n    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n        z.writestr('READ_ME_FIRST.md',f'''# GTSRB - kết quả thực nghiệm\n\nTrạng thái: hoàn tất, đã tính lại chỉ số từ dự đoán từng ảnh.\nPhạm vi: {len(protocol['sample_ids']):,} ảnh mỗi lượt, {len(runs)} lượt.\n{'Đây là full-test chính thức.' if protocol['subset_id']=='official_test' else 'Đây là subset, không phải full-test 12.630 ảnh.'}\n\n## Đọc theo thứ tự\n1. `01_summary/aggregate.csv`: bảng kết quả trung bình và độ lệch chuẩn.\n2. `02_analysis/`: F1 từng lớp, các cặp nhầm, trường hợp được giúp/bị làm sai và biểu đồ.\n3. `03_predictions/test.csv`: toàn bộ dự đoán gộp thành một bảng; lọc theo run_id, method, condition và seed.\n4. `04_protocol/`: cấu hình khóa trước test, trạng thái lượt chạy và thông tin nguồn.\n5. `05_training/`: checkpoint do lần chạy tạo ra và lịch sử huấn luyện, nếu có.\n6. `06_reproduce/`: mã nguồn/notebook kèm theo, nếu có.\n7. `07_supplemental/`: validation/ablation cũ hoặc bổ sung, nếu có; không trộn với kết quả test chính.\n8. `08_processing_extension/`: nghiên cứu khử nhiễu và thứ tự xử lý, nếu đã chạy; tách rõ subset và full-test.\n\n## Quy ước\nNoise seed được lấy trung bình trong mỗi training seed trước khi tính mean và sample SD.\nCác checkpoint dùng chung ảnh; không coi các seed là những ảnh độc lập.\nConfusion matrix có hàng là nhãn thật, cột là lớp dự đoán (0-42).\n`confusion_matrices.npz` gồm run_ids và matrices, đọc bằng numpy.load(..., allow_pickle=False).\nSHA256.json kiểm tra tính toàn vẹn từng file, không tự chứng minh quyền dữ liệu hoặc tính đúng của nhãn.\n\n## Những thứ không đưa vào ZIP\nKhông kèm ảnh GTSRB gốc, trọng số enhancer của tác giả hoặc cache trung gian.\nKhông kèm hàng nghìn ảnh demo lặp; giữ bảng ID/dự đoán để truy vết và tái tạo bằng mã nguồn.\nCác đường dẫn output_path trong bảng là cache tạm có thể đã giải phóng, không phải file bị thiếu trong ZIP.\n''')\n        z.writestr('04_protocol/archive_info.json',json.dumps(metadata,ensure_ascii=False,indent=2))\n        summaries=[r for r,_,_ in runs]\n        z.writestr('01_summary/aggregate.csv',csv_bytes(aggregate(summaries)))\n        z.writestr('01_summary/runs.csv',csv_bytes(summaries))\n        first_meta=next(iter(runs))[0]\n        extra=['run_id','condition','method','train_seed']\n        fields=extra+[k for k in runs[0][1][0] if k not in extra]\n        with z.open('03_predictions/test.csv','w') as binary:\n            stream=io.TextIOWrapper(binary,encoding='utf-8',newline='')\n            writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()\n            for run,predictions,_ in runs:\n                for row in predictions:\n                    writer.writerow({**row,**{k:run[k] for k in extra}})\n            stream.flush();stream.detach()\n        matrices=io.BytesIO()\n        np.savez_compressed(matrices,run_ids=np.asarray([r['run_id'] for r,_,_ in runs]),\n                            matrices=np.asarray([s['confusion_matrix'] for _,_,s in runs],dtype=np.int32))\n        z.writestr('02_analysis/confusion_matrices.npz',matrices.getvalue())\n        for name in ['protocol.json','runs.json','environment.json','classifier_latency.json']:\n            if (folder/name).is_file():z.write(folder/name,'04_protocol/'+name)\n        for name in ['restore.json','checkpoint_hashes.json','upstream_sources.json','source_hashes.json']:\n            if (folder.parent/name).is_file():z.write(folder.parent/name,'04_protocol/'+name)\n        if analysis:\n            analysis=Path(analysis)\n            for name in ['per_class_aggregate.csv','transitions.csv','confusion_pairs.csv','examples.csv','audit.json']:\n                if (analysis/name).is_file():z.write(analysis/name,'02_analysis/'+name)\n            for p in sorted(analysis.glob('*.png')):\n                z.write(p,'02_analysis/figures/'+p.name)\n        train=folder.parent/'train'\n        if train.is_dir():\n            for p in sorted(train.iterdir()):\n                if p.is_file() and (p.suffix in {'.json','.csv'} or (include_checkpoints and p.suffix=='.pt')):\n                    z.write(p,'05_training/'+p.name)\n        validation=folder.parent/'validation'\n        if validation.is_dir():\n            for p in sorted(validation.glob('*.json')):z.write(p,'04_protocol/validation/'+p.name)\n            for p in sorted(validation.glob('*.csv')):z.write(p,'07_supplemental/validation/'+p.name)\n        # Preserve supplemental predictions in one CSV per section rather than hundreds of files.\n        sections=[]\n        if (validation/'predictions').is_dir():sections.append(('validation',validation))\n        if supplemental and (Path(supplemental)/'predictions').is_dir():sections.append(('ablation',Path(supplemental)))\n        if subset and (Path(subset)/'predictions').is_dir():sections.append(('subset',Path(subset)))\n        for label,section in sections:\n            summary_path=section/('validation_runs.csv' if label=='validation' else 'summary.csv')\n            index={}\n            if summary_path.exists():\n                from src.common import config_id\n                for r in read_csv(summary_path):\n                    if 'run_id' in r: rid=r['run_id']\n                    else:\n                        # Validation hashes are generated before CSV serialization.\n                        raw=dict(r)\n                        for k in ['noise_seed','train_seed','n_images']:raw[k]=int(raw[k])\n                        for k in ['macro_f1','accuracy','weighted_f1']:raw[k]=float(raw[k])\n                        rid=config_id(raw)\n                    index[rid]=r\n            paths=sorted((section/'predictions').glob('*.csv'))\n            if not paths:continue\n            first=read_csv(paths[0])[0]\n            meta_fields=['run_id','variant','method','condition','train_seed']\n            fields=meta_fields+[k for k in first if k not in meta_fields]\n            with z.open('07_supplemental/'+label+'/predictions.csv','w') as binary:\n                stream=io.TextIOWrapper(binary,encoding='utf-8',newline='')\n                w=csv.DictWriter(stream,fieldnames=fields);w.writeheader()\n                for p in paths:\n                    meta=index.get(p.stem,{})\n                    for row in read_csv(p):w.writerow({**row,**{k:meta.get(k,p.stem if k=='run_id' else '') for k in meta_fields}})\n                stream.flush();stream.detach()\n            if label in {'ablation','subset'}:\n                for p in sorted(section.glob('*.json')):z.write(p,'07_supplemental/'+label+'/'+p.name)\n                for p in sorted(section.glob('*.csv')):z.write(p,'07_supplemental/'+label+'/'+p.name)\n                if (section/'demo/predictions.csv').exists():z.write(section/'demo/predictions.csv','07_supplemental/demo_predictions.csv')\n        if extension:\n            extension=Path(extension)\n            for scope in ['subset','fulltest']:\n                section=extension/scope\n                if not (section/'summary.csv').exists():continue\n                ep,eruns=audited_runs(section)\n                prefix='08_processing_extension/'+scope+'/'\n                z.writestr(prefix+'aggregate.csv',csv_bytes(aggregate([r for r,_,_ in eruns])))\n                for name in ['protocol.json','runs.json','summary.csv','reuse.json']:\n                    if (section/name).is_file():z.write(section/name,prefix+name)\n                extra=['run_id','condition','method','train_seed']\n                fields=extra+[k for k in eruns[0][1][0] if k not in extra]\n                with z.open(prefix+'predictions.csv','w') as binary:\n                    stream=io.TextIOWrapper(binary,encoding='utf-8',newline='')\n                    writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()\n                    for run,predictions,_ in eruns:\n                        for row in predictions:writer.writerow({**row,**{k:run[k] for k in extra}})\n                    stream.flush();stream.detach()\n                matrices=io.BytesIO()\n                np.savez_compressed(matrices,run_ids=np.asarray([r['run_id'] for r,_,_ in eruns]),\n                    matrices=np.asarray([s['confusion_matrix'] for _,_,s in eruns],dtype=np.int32))\n                z.writestr(prefix+'confusion_matrices.npz',matrices.getvalue())\n                for name in ['per_class_aggregate.csv','transitions.csv','confusion_pairs.csv','examples.csv','order_comparisons.csv','audit.json']:\n                    p=section/'error_analysis'/name\n                    if p.is_file():z.write(p,prefix+'analysis/'+name)\n            validation=extension/'validation'\n            for p in sorted(validation.glob('*.json')):z.write(p,'08_processing_extension/validation/'+p.name)\n            for p in sorted(validation.glob('*.csv')):z.write(p,'08_processing_extension/validation/'+p.name)\n            paths=sorted((validation/'predictions').glob('*.csv'))\n            if paths:\n                fields=['run_id']+[k for k in read_csv(paths[0])[0] if k!='run_id']\n                with z.open('08_processing_extension/validation/predictions.csv','w') as binary:\n                    stream=io.TextIOWrapper(binary,encoding='utf-8',newline='')\n                    writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()\n                    for p in paths:\n                        for row in read_csv(p):writer.writerow({**row,'run_id':p.stem})\n                    stream.flush();stream.detach()\n        if source:\n            source=Path(source)\n            buffer=io.BytesIO()\n            with zipfile.ZipFile(buffer,'w',zipfile.ZIP_DEFLATED) as snapshot:\n                for directory in ['src','configs']:\n                    for p in sorted((source/directory).rglob('*')):\n                        if p.is_file() and p.suffix in {'.py','.json'} and '__pycache__' not in p.parts:\n                            # Only own implementation; upstream external/ is never bundled.\n                            snapshot.write(p,p.relative_to(source).as_posix())\n            z.writestr('06_reproduce/project_source.zip',buffer.getvalue())\n        if notebook:z.write(notebook,'06_reproduce/notebook.ipynb')\n    with zipfile.ZipFile(temporary,'a',compression=zipfile.ZIP_DEFLATED) as z:\n        hashes={}\n        for name in z.namelist():\n            h=hashlib.sha256()\n            with z.open(name) as stream:\n                for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)\n            hashes[name]=h.hexdigest()\n        z.writestr('SHA256.json',json.dumps(hashes,indent=2))\n    temporary.replace(output)\n    return output\n\n\nif __name__=='__main__':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--input',type=Path,required=True)\n    p.add_argument('--output',type=Path,required=True)\n    p.add_argument('--analysis',type=Path)\n    p.add_argument('--source',type=Path)\n    p.add_argument('--notebook',type=Path)\n    p.add_argument('--supplemental',type=Path)\n    p.add_argument('--subset',type=Path)\n    a=p.parse_args()\n    print(pack(a.input,a.output,a.analysis,a.source,a.notebook,a.supplemental,subset=a.subset))\n",
  "src/tools/checkpoint_restore.py": "\"\"\"Restore numeric artifacts from a results ZIP, never execute bundled source.\"\"\"\nfrom collections import defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport torch\n\nfrom src.common import config_id,digest,write_json\nfrom src.experiment import manifest_rows,source_code_hash\nfrom src.model import cnn_config\nfrom .research_evidence import audited_runs,read_csv\nfrom src.metrics import classification\n\n\ndef _split_predictions(z,name,target,allowed,expected_sha=None):\n    \"\"\"Verify the merged CSV while splitting it, with one output file open at a time.\"\"\"\n    h=hashlib.sha256()\n    class Reader(io.RawIOBase):\n        def readable(self):return True\n        def readinto(self,buffer):\n            data=binary.read(len(buffer));h.update(data);buffer[:len(data)]=data;return len(data)\n    current=None;stream=None;written=set()\n    try:\n        with z.open(name) as binary,io.TextIOWrapper(io.BufferedReader(Reader()),encoding='utf-8-sig',newline='') as text:\n            for row in csv.DictReader(text):\n                rid=row['run_id']\n                if rid not in allowed:raise ValueError('Unexpected prediction run')\n                if rid!=current:\n                    if rid in written:raise ValueError('Noncontiguous/duplicate prediction run')\n                    if stream is not None:stream.close()\n                    current=rid;written.add(rid)\n                    path=Path(target)/'predictions'/f'{rid}.csv';path.parent.mkdir(parents=True,exist_ok=True)\n                    stream=path.open('w',encoding='utf-8',newline='')\n                    fields=[k for k in row if k not in {'run_id','variant','condition','method','train_seed'}]\n                    writer=csv.DictWriter(stream,fieldnames=fields,extrasaction='ignore');writer.writeheader()\n                writer.writerow(row)\n    finally:\n        if stream is not None:stream.close()\n    if written!=allowed:raise ValueError('Missing prediction runs')\n    if expected_sha and h.hexdigest()!=expected_sha:raise ValueError('Prediction checksum mismatch')\n\n\ndef restore(root,archive,output,config):\n    root,archive,output=Path(root),Path(archive),Path(output)\n    _,manifest_hash=manifest_rows(root)\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)):raise ValueError('Duplicate archive paths')\n        hashes=json.loads(z.read('SHA256.json')) if 'SHA256.json' in names else {}\n        def read(name):\n            data=z.read(name)\n            if hashes and hashlib.sha256(data).hexdigest()!=hashes.get(name):\n                raise ValueError('Archive checksum mismatch: '+name)\n            return data\n        def save(name,path):\n            data=read(name);path=Path(path)\n            if path.exists() and path.read_bytes()!=data:raise ValueError('Refusing to overwrite different artifact: '+str(path))\n            path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(data)\n        checkpoints=[]\n        for seed in [11,22,33]:\n            candidates=[n for n in [f'05_training/cnn_{seed}.pt',f'notebook_A/train/cnn_{seed}.pt'] if n in names]\n            if len(candidates)!=1:raise ValueError(f'Need exactly one checkpoint for seed {seed}')\n            data=read(candidates[0])\n            saved=torch.load(io.BytesIO(data),map_location='cpu',weights_only=True)\n            if (saved['seed']!=seed or saved['manifest_sha256']!=manifest_hash\n                or cnn_config(saved.get('cnn_config'))!=cnn_config(config['cnn']) or saved.get('pilot',False)):\n                raise ValueError('Checkpoint seed/data/config incompatible')\n            path=output/'train'/f'cnn_{seed}.pt';save(candidates[0],path);checkpoints.append(path)\n        selection_names=[n for n in ['04_protocol/validation/selection.json','notebook_A/validation/selection.json'] if n in names]\n        selection_ready=False\n        if len(selection_names)==1:\n            selection=json.loads(read(selection_names[0]))\n            if set(selection['checkpoints'].values())!={digest(p) for p in checkpoints}:\n                raise ValueError('Validation/checkpoint hashes differ')\n            if (selection['manifest_sha256']==manifest_hash and selection['code_hash']==source_code_hash()\n                and selection.get('status')=='completed' and not selection.get('pilot',True)):\n                selection['checkpoints']={str(p.resolve()):digest(p) for p in checkpoints}\n                target=output/'validation/selection.json'\n                if target.exists() and json.loads(target.read_text())!=selection:raise ValueError('Existing selection differs')\n                write_json(target,selection);selection_ready=True\n                for name,destination in [\n                    ('07_supplemental/validation/validation_runs.csv','validation/validation_runs.csv'),\n                    ('07_supplemental/validation/predictions.csv','validation/restored_predictions.csv')]:\n                    if name in names:save(name,output/destination)\n        for seed in [11,22,33]:\n            for filename in [f'train_{seed}.csv',f'learning_{seed}.csv',f'train_{seed}.json',f'validation_{seed}.json']:\n                candidates=[n for n in ['05_training/'+filename,'notebook_A/train/'+filename] if n in names]\n                if len(candidates)==1:save(candidates[0],output/'train'/filename)\n        recovered=[]\n        # Readable archive: restore merged predictions to the ordinary audited run layout.\n        for scope,prefix,summary,predictions in [\n            ('fulltest','04_protocol/','01_summary/runs.csv','03_predictions/test.csv'),\n            ('subset','07_supplemental/subset/','07_supplemental/subset/summary.csv','07_supplemental/subset/predictions.csv')]:\n            if prefix+'protocol.json' not in names or summary not in names or predictions not in names:continue\n            protocol=json.loads(read(prefix+'protocol.json'))\n            if (protocol['manifest_sha256']!=manifest_hash or protocol['code_hash']!=source_code_hash()\n                or {c['sha256'] for c in protocol['checkpoints']}!={digest(p) for p in checkpoints}):continue\n            actual='fulltest' if protocol['subset_id']=='official_test' else 'subset'\n            target=output/actual\n            save(prefix+'protocol.json',target/'protocol.json');save(prefix+'runs.json',target/'runs.json')\n            save(summary,target/'summary.csv')\n            allowed={r['run_id'] for r in csv.DictReader(io.StringIO(read(summary).decode('utf-8-sig')))}\n            if allowed!={config_id(r) for r in protocol['runs']}:\n                raise ValueError('Prediction run IDs differ from frozen protocol')\n            _split_predictions(z,predictions,target,allowed,hashes.get(predictions))\n            audited_runs(target);recovered.append(actual)\n        ablation_ready=False\n        prefix='07_supplemental/ablation/'\n        if all(prefix+n in names for n in ['protocol.json','summary.csv','predictions.csv']):\n            protocol=json.loads(read(prefix+'protocol.json'))\n            if set(protocol['checkpoint_hashes'].values())!={digest(p) for p in checkpoints}:\n                raise ValueError('Ablation checkpoint mismatch')\n            target=output/'ablation';save(prefix+'protocol.json',target/'protocol.json');save(prefix+'summary.csv',target/'summary.csv')\n            summaries=read_csv(target/'summary.csv');allowed={r['run_id'] for r in summaries}\n            for r in summaries:\n                run={k:r[k] for k in ['variant','method','noise_seed','train_seed']}\n                run['noise_seed']=int(run['noise_seed']);run['train_seed']=int(run['train_seed'])\n                if config_id(run)!=r['run_id']:raise ValueError('Invalid ablation run ID')\n            if len(allowed)!=288:raise ValueError('Incomplete ablation matrix')\n            _split_predictions(z,prefix+'predictions.csv',target,allowed,hashes.get(prefix+'predictions.csv'))\n            for r in summaries:\n                rows=read_csv(target/'predictions'/f\"{r['run_id']}.csv\")\n                if len(rows)!=430 or {p['sample_id'] for p in rows}!=set(protocol['sample_ids']):raise ValueError('Ablation coverage differs')\n                if any(p['noise_seed']!=r['noise_seed'] for p in rows):raise ValueError('Ablation noise seed differs')\n                score=classification([int(p['class_id']) for p in rows],[int(p['prediction']) for p in rows])\n                if any(abs(score[k]-float(r[k]))>1e-8 for k in ['accuracy','macro_f1','weighted_f1']):raise ValueError('Ablation metric differs')\n            ablation_ready=True\n    record={'archive_sha256':digest(archive),'new_training':False,'selection_reusable':selection_ready,\n        'baseline_scopes':sorted(set(recovered)),'ablation_reusable':ablation_ready,'checkpoints':{str(p):digest(p) for p in checkpoints},\n        'bundled_source_executed':False,'note':'Changed baseline implementation requires validation again, never automatic retraining.'}\n    write_json(output/'restore.json',record)\n    return checkpoints,record\n",
  "src/tools/processing_pipeline.py": "\"\"\"Denoising and ordered RGB pipelines. No operation receives labels or references.\"\"\"\nimport cv2\nimport numpy as np\n\nfrom src.common import validate_image, digest\nfrom src.enhancement import Enhancer, provenance\nfrom src.experiment import source_code_hash\n\n\ndef _positive(value, name):\n    if not np.isfinite(value) or value <= 0:\n        raise ValueError(f'{name} must be finite and positive')\n\n\ndef _odd(value, name):\n    if not isinstance(value, int) or isinstance(value, bool) or value < 1 or value % 2 != 1:\n        raise ValueError(f'{name} must be a positive odd integer')\n\n\ndef denoise(image, method, params):\n    image = validate_image(image)\n    if method == 'gaussian':\n        _odd(params['kernel'], 'kernel'); _positive(params['sigma'], 'sigma')\n        result = cv2.GaussianBlur(image, (params['kernel'], params['kernel']), params['sigma'], borderType=cv2.BORDER_REFLECT_101)\n    elif method == 'bilateral':\n        _odd(params['diameter'], 'diameter')\n        _positive(params['sigma_color'], 'sigma_color'); _positive(params['sigma_space'], 'sigma_space')\n        result = cv2.bilateralFilter(image, params['diameter'], params['sigma_color'], params['sigma_space'], borderType=cv2.BORDER_REFLECT_101)\n    elif method == 'nlm':\n        _positive(params['h'], 'h'); _positive(params['h_color'], 'h_color')\n        _odd(params['template'], 'template'); _odd(params['search'], 'search')\n        if params['search'] < params['template']:\n            raise ValueError('search must be at least template')\n        # OpenCV's color NLM expects uint8 BGR and internally uses Lab.\n        bgr = cv2.cvtColor(np.rint(image * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)\n        result = cv2.fastNlMeansDenoisingColored(bgr, None, params['h'], params['h_color'], params['template'], params['search'])\n        result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB).astype(np.float32) / 255\n    else:\n        raise ValueError(f'Unknown denoiser: {method}')\n    if result.shape != image.shape:\n        raise ValueError('Denoiser changed image dimensions')\n    return validate_image(np.clip(result, 0, 1).astype(np.float32))\n\n\nclass Pipeline:\n    def __init__(self, spec, device):\n        self.spec = spec\n        self.steps = []\n        for step in spec['steps']:\n            if step['kind'] == 'denoise':\n                if step['method'] not in {'gaussian','bilateral','nlm'}:\n                    raise ValueError('Unknown denoiser')\n                self.steps.append((step, None))\n            elif step['kind'] == 'enhance':\n                self.steps.append((step, Enhancer(step['method'], step['params'], device)))\n            else:\n                raise ValueError('Unknown pipeline step kind')\n\n    def __call__(self, image, sample_id=''):\n        result = validate_image(image).copy()\n        shape = result.shape\n        for step, enhancer in self.steps:\n            result = denoise(result, step['method'], step['params']) if enhancer is None else enhancer(result, sample_id)\n        if result.shape != shape:\n            raise ValueError('Pipeline changed dimensions')\n        return validate_image(result)\n\n\ndef pipeline_provenance(spec):\n    operations = []\n    for step in spec['steps']:\n        if step['kind'] == 'enhance':\n            operations.append(provenance(step['method'], step['params']))\n        elif step['kind'] == 'denoise':\n            operations.append({'method': step['method'], 'params': step['params'],\n                               'quantization': 'uint8 BGR to Lab inside OpenCV' if step['method']=='nlm' else 'float32 RGB',\n                               'opencv': cv2.__version__})\n        else:\n            raise ValueError('Unknown pipeline step')\n    return {'spec': spec, 'operations': operations, 'baseline_implementation': source_code_hash(),\n            'pipeline_implementation': digest(__file__), 'reference_access': False, 'label_access': False}\n",
  "src/tools/processing_study.py": "\"\"\"Validation-first study of denoising, enhancement and operation order.\"\"\"\nfrom collections import defaultdict\nfrom copy import deepcopy\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nfrom src.common import (config_id,digest,read_csv,write_csv,write_json,read_image,synchronize)\nfrom src.data import subset_rows\nfrom src.degradation import degrade\nfrom src.experiment import load_cnn,manifest_rows,source_code_hash\nfrom src.enhancement import provenance\nfrom src.metrics import classification,image_metrics\nfrom src.model import image_tensor\nfrom .processing_pipeline import Pipeline,pipeline_provenance\n\n\ndef study_hash():\n    return config_id({p.name:digest(p) for p in [Path(__file__),Path(__file__).with_name('processing_pipeline.py')]})\n\n\ndef _enhance(method,params):\n    return {'kind':'enhance','method':method,'params':deepcopy(params)}\n\n\ndef _denoise(method,params):\n    return {'kind':'denoise','method':method,'params':deepcopy(params)}\n\n\ndef controls(baseline):\n    result={'identity':{'steps':[]}}\n    for method in dict.fromkeys(['gamma','clahe',baseline['chosen_method']]):\n        if method!='identity':result[method]={'steps':[_enhance(method,baseline['enhancers'][method])]}\n    return result\n\n\ndef choose_fulltest_methods(scores):\n    \"\"\"Select matched orders, their standalone controls and winners using validation only.\"\"\"\n    points={r['method']:r for r in scores}\n    base=sorted(r['method'] for r in scores if r['category']=='control')\n    singles=sorted((r for r in scores if r['category']=='denoise'),key=lambda r:(-r['macro_f1'],r['method']))\n    pairs=[]\n    for r in scores:\n        if r['category']=='denoise_then_enhance':\n            mate='post__'+r['method'].removeprefix('pre__')\n            pairs.append(((r['macro_f1']+points[mate]['macro_f1'])/2,r['method'],mate,r['denoiser']))\n    if not pairs or not singles:raise ValueError('Missing controlled order pairs or denoisers')\n    best=sorted(pairs,key=lambda x:(-x[0],x[1]))[0]\n    overall=sorted(scores,key=lambda r:(-r['macro_f1'],r['latency_ms'],r['method']))[0]['method']\n    return list(dict.fromkeys(base+[singles[0]['method'],'denoise__'+best[3],best[1],best[2],overall]))\n\n\ndef _reference_images(root,rows):\n    \"\"\"Decode each clean PNG once per scope; keep read-only float32 RGB in RAM.\"\"\"\n    references={}\n    for row in rows:\n        image=read_image(Path(root)/row['crop_path']);image.setflags(write=False)\n        references[row['sample_id']]=image\n    print('Clean RGB references in RAM:',len(references),'images,',\n          round(sum(a.nbytes for a in references.values())/1024**2,1),'MiB',flush=True)\n    return references\n\n\ndef _process_group(root,rows,config,condition,noise,spec,models,device,references=None):\n    \"\"\"Stream enhanced RGB batches to all three CNNs without intermediate image files.\"\"\"\n    def reference(row):\n        return references[row['sample_id']] if references is not None else read_image(Path(root)/row['crop_path'])\n    pipeline=Pipeline(spec,device)\n    if condition!='clean':\n        first=rows[0]\n        warm=degrade(reference(first),**config['conditions'][condition],\n                     noise_seed=int(noise),sample_id=first['sample_id'],condition=condition)\n        for _ in range(2):pipeline(warm,first['sample_id'])\n        synchronize(device)\n    predictions={saved['seed']:[] for _,saved in models}\n    batch_size=config['cnn']['batch_size']\n    for start in range(0,len(rows),batch_size):\n        batch=rows[start:start+batch_size]\n        tensors,records=[],[]\n        for row in batch:\n            clean=reference(row)\n            dark=clean if condition=='clean' else degrade(clean,**config['conditions'][condition],\n                noise_seed=int(noise),sample_id=row['sample_id'],condition=condition)\n            synchronize(device);began=time.perf_counter()\n            enhanced=clean if condition=='clean' else pipeline(dark,row['sample_id'])\n            synchronize(device)\n            latency=0.0 if condition=='clean' else 1000*(time.perf_counter()-began)\n            tensors.append(image_tensor(enhanced))\n            records.append({'sample_id':row['sample_id'],'group_id':row['group_id'],\n                'source_id':row['source_id'],'class_id':int(row['class_id']),'noise_seed':noise,\n                **image_metrics(clean,enhanced,config['ssim']),'latency_ms':latency,'output_path':''})\n        images=torch.stack(tensors).to(device)\n        with torch.inference_mode():\n            for model,saved in models:\n                probabilities=model(images).softmax(1).cpu().numpy()\n                if not np.isfinite(probabilities).all():raise ValueError('Nonfinite classifier probabilities')\n                for record,p in zip(records,probabilities):\n                    predictions[saved['seed']].append({**record,'prediction':int(p.argmax()),\n                        'true_probability':float(p[record['class_id']])})\n    return predictions\n\n\ndef _score(rows):\n    return classification([int(r['class_id']) for r in rows],[int(r['prediction']) for r in rows])\n\n\ndef tune(root,baseline_path,checkpoint_paths,extension_config,device,output):\n    root,output=Path(root),Path(output)\n    baseline=json.loads(Path(baseline_path).read_text())\n    plan=json.loads(Path(extension_config).read_text())\n    all_rows,manifest_hash=manifest_rows(root)\n    if baseline['manifest_sha256']!=manifest_hash or baseline['code_hash']!=source_code_hash():\n        raise ValueError('Baseline selection is stale; reuse CNNs but revalidate the baseline first')\n    if baseline.get('pilot',True) or baseline.get('status')!='completed':raise ValueError('Need completed baseline validation')\n    if {digest(p) for p in checkpoint_paths}!=set(baseline['checkpoints'].values()):raise ValueError('Checkpoint mismatch')\n    for method,params in baseline['enhancers'].items():\n        if provenance(method,params)!=baseline['enhancer_provenance'][method]:raise ValueError('Baseline enhancer changed')\n    lock={'study_hash':study_hash(),'baseline_selection_sha256':digest(baseline_path),\n          'plan_sha256':digest(extension_config),'manifest_sha256':manifest_hash,'code_hash':source_code_hash()}\n    if (output/'validation_lock.json').exists() and json.loads((output/'validation_lock.json').read_text())!=lock:\n        raise ValueError('Validation output belongs to other inputs/code')\n    write_json(output/'validation_lock.json',lock)\n    if (output/'selection.json').exists():\n        previous=json.loads((output/'selection.json').read_text())\n        if previous['study_hash']==study_hash() and previous['baseline_selection_sha256']==digest(baseline_path) and previous['plan_sha256']==digest(extension_config):\n            return previous\n        raise ValueError('Extension selection belongs to different inputs/code; use a new output')\n    rows=subset_rows(all_rows,'validation',plan['validation_per_class'])\n    if {r['sample_id'] for r in rows}!=set(baseline['sample_ids']):raise ValueError('Validation subset differs from baseline')\n    models=[load_cnn(p,manifest_hash,device,baseline['config']['cnn']) for p in checkpoint_paths]\n    if {s['seed'] for _,s in models}!={11,22,33}:raise ValueError('Require classifier seeds 11/22/33')\n    cfg=deepcopy(baseline['config']);cfg['enhancers'].update(baseline['enhancers'])\n    references=_reference_images(root,rows)\n    registry=controls(baseline);scores=[];details=[]\n    def measure(name,spec,category,denoiser=''):\n        values=[];latencies=[]\n        candidate_id=config_id(spec)\n        for condition in cfg['conditions']:\n            for noise in cfg['noise_seeds']:\n                planned={s['seed']:{'method':name,'candidate_id':candidate_id,'condition':condition,'noise_seed':noise,'train_seed':s['seed']} for _,s in models}\n                predictions={}\n                for seed,run in planned.items():\n                    rid=config_id(run);record=output/'completed'/f'{rid}.json';path=output/'predictions'/f'{rid}.csv'\n                    if record.exists():\n                        if not path.exists() or digest(path)!=json.loads(record.read_text())['prediction_sha256']:\n                            raise ValueError('Completed validation predictions changed')\n                        predictions[seed]=_verified_predictions(path,rows,run)\n                if len(predictions)!=len(models):\n                    missing=[m for m in models if m[1]['seed'] not in predictions]\n                    predictions.update(_process_group(root,rows,cfg,condition,noise,spec,missing,device,references))\n                latencies.append(np.mean([float(r['latency_ms']) for r in next(iter(predictions.values()))]))\n                for seed,preds in predictions.items():\n                    score=_score(preds)\n                    run={'method':name,'candidate_id':candidate_id,'condition':condition,'noise_seed':noise,'train_seed':seed}\n                    rid=config_id(run)\n                    write_csv(output/'predictions'/f'{rid}.csv',preds)\n                    write_json(output/'completed'/f'{rid}.json',{'prediction_sha256':digest(output/'predictions'/f'{rid}.csv')})\n                    details.append({**run,'run_id':rid,'n_images':len(rows),**{k:score[k] for k in ['accuracy','macro_f1','weighted_f1']}})\n                    values.append(score['macro_f1'])\n                write_csv(output/'summary.csv',details)\n        result={'method':name,'category':category,'denoiser':denoiser,'candidate_id':candidate_id,\n                'macro_f1':float(np.mean(values)),'latency_ms':float(np.mean(latencies)),'spec':spec,'origin':'extension_validation'}\n        print('Extension validation:',name,candidate_id,result['macro_f1'],flush=True)\n        return result\n    baseline_points={r['method']:r for r in baseline['scores']}\n    for name,spec in registry.items():\n        r=baseline_points[name]\n        scores.append({'method':name,'category':'control','denoiser':'','macro_f1':r['macro_f1'],\n                       'latency_ms':r['latency_ms'],'spec':spec,'origin':'same_frozen_baseline_validation'})\n    chosen_strengths={}\n    for method,parameters in plan['denoisers'].items():\n        candidates=[measure('denoise__'+method,{'steps':[_denoise(method,p)]},'denoise',method) for p in parameters]\n        selected=sorted(candidates,key=lambda r:(-r['macro_f1'],r['candidate_id']))[0]\n        scores.append(selected);registry[selected['method']]=selected['spec']\n        chosen_strengths[method]=selected['spec']['steps'][0]['params']\n        write_json(output/f'strength_{method}.json',candidates)\n    # Fix one strength for each denoiser before comparing the two orders.\n    for denoiser,parameters in chosen_strengths.items():\n        for enhancer in plan['composition_enhancers']:\n            d=_denoise(denoiser,parameters);e=_enhance(enhancer,baseline['enhancers'][enhancer])\n            for prefix,steps,category in [('pre',[d,e],'denoise_then_enhance'),('post',[e,d],'enhance_then_denoise')]:\n                name=f'{prefix}__{denoiser}__{enhancer}';spec={'steps':steps}\n                registry[name]=spec;scores.append(measure(name,spec,category,denoiser))\n    full_methods=choose_fulltest_methods(scores)\n    chosen=sorted(scores,key=lambda r:(-r['macro_f1'],r['latency_ms'],r['method']))[0]['method']\n    selected={'study':'denoise_order_extension','status':'completed','pilot':False,'study_hash':study_hash(),\n        'code_hash':source_code_hash(),'baseline_selection_sha256':digest(baseline_path),'plan_sha256':digest(extension_config),\n        'manifest_sha256':manifest_hash,'sample_ids':[r['sample_id'] for r in rows],'config':cfg,\n        'checkpoints':{str(Path(p).resolve()):digest(p) for p in checkpoint_paths},'registry':registry,'scores':scores,\n        'chosen_method':chosen,'fulltest_methods':full_methods,'methods':list(registry),'plan':plan,\n        'pipeline_provenance':{name:pipeline_provenance(spec) for name,spec in registry.items()},\n        'selection_rule':'Fixed strengths for order comparisons; paired mean validation score selects full-test order pair; no test selection.'}\n    write_json(output/'selection.json',selected)\n    return selected\n\n\ndef freeze(root,selection_path,output,per_class=10):\n    root,output=Path(root),Path(output)\n    selection=json.loads(Path(selection_path).read_text())\n    all_rows,manifest_hash=manifest_rows(root)\n    if selection['study_hash']!=study_hash() or selection['code_hash']!=source_code_hash() or selection['manifest_sha256']!=manifest_hash:\n        raise ValueError('Extension selection/code/data changed')\n    if selection.get('pilot',True) or selection['status']!='completed':raise ValueError('Incomplete/pilot selection')\n    rows=subset_rows(all_rows,'test',per_class,42)\n    methods=selection['methods'] if per_class is not None else selection['fulltest_methods']\n    checkpoints=[]\n    for path,sha in selection['checkpoints'].items():\n        if digest(path)!=sha:raise ValueError('Checkpoint changed')\n        saved=torch.load(path,map_location='cpu',weights_only=True)\n        checkpoints.append({'path':path,'sha256':sha,'seed':saved['seed']})\n    if {p['seed'] for p in checkpoints}!={11,22,33}:raise ValueError('Missing classifier seed')\n    runs=[]\n    for c in checkpoints:\n        runs.append({'condition':'clean','noise_seed':'','method':'clean','train_seed':c['seed']})\n        for condition in selection['config']['conditions']:\n            for noise in selection['config']['noise_seeds']:\n                for method in methods:runs.append({'condition':condition,'noise_seed':noise,'method':method,'train_seed':c['seed']})\n    protocol={'study':'denoise_order_extension','pilot':False,'selection_sha256':digest(selection_path),\n        'selection':{**selection,'methods':methods},'sample_ids':[r['sample_id'] for r in rows],\n        'subset_id':'official_test' if per_class is None else config_id([r['sample_id'] for r in rows]),\n        'manifest_sha256':manifest_hash,'config':selection['config'],'study_hash':study_hash(),\n        'code_hash':source_code_hash(),'checkpoints':checkpoints,'runs':runs,\n        'registry':{name:selection['registry'][name] for name in methods},\n        'pipeline_provenance':{name:selection['pipeline_provenance'][name] for name in methods}}\n    if output.exists():\n        if json.loads(output.read_text())!=protocol:raise ValueError('Frozen extension protocol differs')\n        return protocol\n    write_json(output,protocol)\n    return protocol\n\n\ndef _verified_predictions(path,rows,run):\n    predictions=read_csv(path)\n    by_id={r['sample_id']:r for r in rows}\n    if len(predictions)!=len(rows) or {r['sample_id'] for r in predictions}!=set(by_id):raise ValueError('Prediction coverage mismatch')\n    for p in predictions:\n        truth=by_id[p['sample_id']]\n        if int(p['class_id'])!=int(truth['class_id']) or p['group_id']!=truth['group_id'] or str(p['noise_seed'])!=str(run['noise_seed']):\n            raise ValueError('Predictions differ from current ground truth/groups/noise')\n    return predictions\n\n\ndef evaluate(root,protocol_path,output,device,baseline_output=None):\n    from .research_evidence import audited_runs\n    root,output=Path(root),Path(output)\n    protocol=json.loads(Path(protocol_path).read_text())\n    all_rows,manifest_hash=manifest_rows(root)\n    if protocol['study_hash']!=study_hash() or protocol['code_hash']!=source_code_hash() or manifest_hash!=protocol['manifest_sha256']:\n        raise ValueError('Frozen extension code/data changed')\n    by_id={r['sample_id']:r for r in all_rows if r['split']=='test'}\n    if len(set(protocol['sample_ids']))!=len(protocol['sample_ids']) or not set(protocol['sample_ids'])<=by_id.keys():raise ValueError('Invalid test IDs')\n    rows=[by_id[i] for i in protocol['sample_ids']]\n    for c in protocol['checkpoints']:\n        if digest(c['path'])!=c['sha256']:raise ValueError('Checkpoint changed')\n    for method,spec in protocol['registry'].items():\n        if pipeline_provenance(spec)!=protocol['pipeline_provenance'][method]:raise ValueError('Pipeline changed')\n    lock=output/'protocol.json'\n    if lock.exists() and json.loads(lock.read_text())!=protocol:raise ValueError('Output belongs to another protocol')\n    write_json(lock,protocol)\n    old_summaries={r['run_id']:r for r in read_csv(output/'summary.csv')} if (output/'summary.csv').exists() else {}\n    old_states={config_id({k:s[k] for k in ['condition','method','noise_seed','train_seed']}):s for s in json.loads((output/'runs.json').read_text())} if (output/'runs.json').exists() else {}\n    states={config_id(r):dict(r,status='planned') for r in protocol['runs']};summaries={}\n    def keep(run,predictions,origin):\n        rid=config_id(run);score=_score(predictions)\n        write_csv(output/'predictions'/f'{rid}.csv',predictions)\n        write_json(output/'classes'/f'{rid}.json',score)\n        summaries[rid]={'dataset':'gtsrb','split':'test','protocol':'processing_extension','subset_id':protocol['subset_id'],\n            **run,'run_id':rid,'config_id':config_id(protocol['registry'].get(run['method'],{'steps':[]})),\n            'n_images':len(rows),**{k:score[k] for k in ['accuracy','macro_f1','weighted_f1']},\n            'psnr_mean':float(np.mean([float(r['psnr']) for r in predictions])),\n            'ssim_mean':float(np.mean([float(r['ssim']) for r in predictions])),\n            'latency_ms':float(np.mean([float(r['latency_ms']) for r in predictions]))}\n        states[rid]=dict(run,status='completed',origin=origin,prediction_sha256=digest(output/'predictions'/f'{rid}.csv'))\n        write_csv(output/'summary.csv',list(summaries.values()));write_json(output/'runs.json',list(states.values()))\n    for run in protocol['runs']:\n        rid=config_id(run);path=output/'predictions'/f'{rid}.csv';state=old_states.get(rid,{})\n        if state.get('status')=='completed' and rid in old_summaries and path.is_file():\n            if digest(path)!=state['prediction_sha256']:raise ValueError('Completed prediction file changed')\n            preds=_verified_predictions(path,rows,run);score=_score(preds)\n            if any(not math.isclose(score[k],float(old_summaries[rid][k]),abs_tol=1e-8,rel_tol=0) for k in ['accuracy','macro_f1','weighted_f1']):raise ValueError('Completed metric changed')\n            summaries[rid]=old_summaries[rid];states[rid]=state\n    reused=[]\n    if baseline_output:\n        baseline,runs=audited_runs(baseline_output)\n        same=(baseline['manifest_sha256']==manifest_hash and baseline['code_hash']==protocol['code_hash']\n            and baseline['config']==protocol['config'] and set(baseline['sample_ids'])==set(protocol['sample_ids'])\n            and {c['seed']:c['sha256'] for c in baseline['checkpoints']}=={c['seed']:c['sha256'] for c in protocol['checkpoints']})\n        if not same:raise ValueError('Baseline reuse requires identical data/config/checkpoints/implementation')\n        lookup={r['run_id']:(r,p) for r,p,_ in runs}\n        for run in protocol['runs']:\n            rid=config_id(run)\n            if rid in summaries or rid not in lookup:continue\n            # Only baseline controls have the same 4-field run ID and same locked enhancer parameters.\n            if run['method']!='clean':\n                expected={'steps':[]} if run['method']=='identity' else {'steps':[_enhance(run['method'],baseline['config']['enhancers'][run['method']])]}\n                if protocol['registry'][run['method']]!=expected:raise ValueError('Baseline control parameters differ')\n            _,preds=lookup[rid]\n            _verified_predictions(Path(baseline_output)/'predictions'/f'{rid}.csv',rows,run)\n            keep(run,preds,'verified_baseline_reuse');reused.append(rid)\n    groups=defaultdict(list)\n    for run in protocol['runs']:\n        if config_id(run) not in summaries:groups[run['condition'],str(run['noise_seed']),run['method']].append(run)\n    models=[]\n    if groups:models=[load_cnn(c['path'],manifest_hash,device,protocol['config']['cnn']) for c in protocol['checkpoints']]\n    references=_reference_images(root,rows) if groups else None\n    for (condition,noise,method),runs in groups.items():\n        needed={r['train_seed'] for r in runs};active=[m for m in models if m[1]['seed'] in needed]\n        spec={'steps':[]} if method=='clean' else protocol['registry'][method]\n        predictions=_process_group(root,rows,protocol['config'],condition,noise,spec,active,device,references)\n        for run in runs:keep(run,predictions[run['train_seed']],'extension_inference')\n        print('Extension test completed:',len(summaries),'/',len(protocol['runs']),method,condition,noise,flush=True)\n    write_csv(output/'summary.csv',[summaries[config_id(r)] for r in protocol['runs']])\n    write_json(output/'runs.json',[states[config_id(r)] for r in protocol['runs']])\n    write_json(output/'reuse.json',{'new_training':False,\n        'reused_run_ids':[rid for rid,s in states.items() if s.get('origin')=='verified_baseline_reuse'],\n        'source':str(baseline_output) if baseline_output else None,\n        'latency_note':'Reused control latency was measured in the baseline session; cross-session timing is descriptive only.'})\n    audited_runs(output)\n    return output\n\n\ndef analyze(folder,output):\n    \"\"\"Standard error analysis plus matched comparison of the two orders.\"\"\"\n    from .research_evidence import analyze as error_analysis,audited_runs\n    from src.metrics import paired_rows\n    error_analysis(folder,output)\n    protocol,runs=audited_runs(folder)\n    index={(r['condition'],r['method'],r['train_seed'],r['noise_seed']):(r,p,s) for r,p,s in runs}\n    comparisons=[]\n    for before,preds,score in runs:\n        if not before['method'].startswith('pre__'):continue\n        mate='post__'+before['method'].removeprefix('pre__')\n        key=(before['condition'],mate,before['train_seed'],before['noise_seed'])\n        if key not in index:continue  # The global winner can be retained outside the selected matched pair.\n        after,other,other_score=index[key]\n        helped=harmed=0\n        for a,b in paired_rows(preds,other):\n            correct_a=a['class_id']==a['prediction'];correct_b=b['class_id']==b['prediction']\n            helped+=int(not correct_a and correct_b);harmed+=int(correct_a and not correct_b)\n        _,denoiser,enhancer=before['method'].split('__')\n        comparisons.append({'condition':before['condition'],'denoiser':denoiser,'enhancer':enhancer,\n            'train_seed':before['train_seed'],'noise_seed':before['noise_seed'],\n            'pre_method':before['method'],'post_method':mate,'n_images':len(preds),\n            'post_minus_pre_accuracy_pp':100*(other_score['accuracy']-score['accuracy']),\n            'post_minus_pre_macro_f1_pp':100*(other_score['macro_f1']-score['macro_f1']),\n            'post_helped':helped,'post_harmed':harmed,\n            'post_minus_pre_psnr':float(after['psnr_mean'])-float(before['psnr_mean']),\n            'post_minus_pre_ssim':float(after['ssim_mean'])-float(before['ssim_mean'])})\n    if comparisons:write_csv(Path(output)/'order_comparisons.csv',comparisons)\n    return output\n",
  "src/tools/baseline_input.py": "\"\"\"Read the pinned ZIP from existing public notebook Output; no Dataset or API key.\"\"\"\nfrom collections import Counter\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import urlparse\nimport zipfile\n\nimport requests\n\nfrom src.common import digest, write_json\n\nSOURCE = 'kieuthithutrang/gtsrb-low-light-enhancement-for-traffic-signs'\nVERSION = 'v1'\nORIGINAL_SHA256 = '1141f3b3be6f0cb59918afa1e685947a9b07aa60e6601c4011386dc88efd1f0b'\nVERIFIED_SHA256 = '01ee0fa7c2b42288c644b0c21d200344ff6cffb5c549f96b4a3abc2c917b6091'\nMANIFEST_SHA256 = '0ea7537e92c03790e49460f12558e6a268a0d6f310084970baf5d2a61dc7fce5'\nOUTPUT_API = 'https://api.kaggle.com/v1/kernels.KernelsApiService/ListKernelSessionOutput'\n\n\ndef download_original(destination):\n    destination = Path(destination)\n    if destination.exists():\n        if digest(destination) != ORIGINAL_SHA256:\n            raise ValueError('Existing baseline ZIP checksum mismatch')\n        return destination\n    owner, slug = SOURCE.split('/')\n    response = requests.post(OUTPUT_API, json=dict(userName=owner, kernelSlug=slug,\n        versionLabel=VERSION, pageSize=100), timeout=60)\n    response.raise_for_status()\n    matches = [row for row in response.json().get('files', []) if row.get('fileName') == 'gtsrb-notebook-results.zip']\n    if len(matches) != 1:\n        raise ValueError('Pinned public notebook Output does not contain exactly one baseline ZIP')\n    url = matches[0]['url']\n    parsed = urlparse(url)\n    if parsed.scheme != 'https' or not parsed.hostname:\n        raise ValueError('Invalid baseline download URL')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_suffix('.partial.zip')\n    h = hashlib.sha256()\n    with requests.get(url, stream=True, timeout=120) as download:\n        download.raise_for_status()\n        with partial.open('wb') as stream:\n            for chunk in download.iter_content(1024 * 1024):\n                h.update(chunk)\n                stream.write(chunk)\n    if h.hexdigest() != ORIGINAL_SHA256:\n        partial.unlink()\n        raise ValueError('Downloaded baseline ZIP checksum mismatch')\n    partial.replace(destination)\n    return destination\n\n\ndef normalize_original(original, destination):\n    \"\"\"Repair only the known duplicate supplemental paths in this exact old export.\"\"\"\n    original, destination = Path(original), Path(destination)\n    if digest(original) != ORIGINAL_SHA256:\n        raise ValueError('Only the pinned original export may be normalized')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_suffix('.partial.zip')\n    with zipfile.ZipFile(original) as source, zipfile.ZipFile(partial, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as output:\n        counts = Counter(source.namelist())\n        prefix = '07_supplemental/ablation/'\n        if {n: c for n, c in counts.items() if c > 1} != {prefix + 'protocol.json': 2, prefix + 'summary.csv': 2}:\n            raise ValueError('Unexpected legacy duplicate paths')\n        expected = json.loads(source.read('SHA256.json'))\n        seen, hashes, moved = Counter(), {}, []\n        for info in source.infolist():\n            name = info.filename\n            if name == 'SHA256.json':\n                continue\n            if PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or '\\\\' in name:\n                raise ValueError('Unsafe baseline archive path')\n            seen[name] += 1\n            target = name\n            if name in [prefix + 'runs.json', prefix + 'worker_devices.json'] or (\n                    name in [prefix + 'protocol.json', prefix + 'summary.csv'] and seen[name] == 2):\n                target = name.replace(prefix, '07_supplemental/subset/', 1)\n                moved.append({'from': name, 'entry': seen[name], 'to': target})\n            if target in hashes:\n                raise ValueError('Unresolved duplicate archive path')\n            h = hashlib.sha256()\n            with source.open(info) as stream, output.open(target, 'w', force_zip64=True) as writer:\n                for chunk in iter(lambda: stream.read(1024 * 1024), b''):\n                    h.update(chunk)\n                    writer.write(chunk)\n            sha = h.hexdigest()\n            if seen[name] == counts[name] and expected.get(name) != sha:\n                raise ValueError('Baseline member checksum mismatch: ' + name)\n            hashes[target] = sha\n        recovery = {'source_archive_sha256': ORIGINAL_SHA256, 'source_preserved': True,\n            'moved_entries': moved,\n            'execution_result': 'Full-test completed; original notebook failed during final subset ZIP rename across mounts.',\n            'changed_predictions': False,\n            'legacy_first_duplicate_integrity': 'ZIP CRC verified; legacy hash map recorded only last duplicate.'}\n        data = (json.dumps(recovery, indent=2) + '\\n').encode()\n        name = '04_protocol/export_recovery.json'\n        output.writestr(name, data)\n        hashes[name] = hashlib.sha256(data).hexdigest()\n        canonical = json.dumps(hashes, sort_keys=True, separators=(',', ':')).encode()\n        if hashlib.sha256(canonical).hexdigest() != MANIFEST_SHA256:\n            raise ValueError('Normalized contents differ from the locally verified baseline')\n        output.writestr('SHA256.json', json.dumps(hashes, indent=2))\n    partial.replace(destination)\n    return destination\n\n\ndef resolve_baseline(folder, archive=None):\n    folder = Path(folder)\n    if archive is None:\n        archive = download_original(folder / 'gtsrb-notebook-results.zip')\n    archive = Path(archive)\n    sha = digest(archive)\n    if sha == VERIFIED_SHA256:\n        result = archive\n    elif sha == ORIGINAL_SHA256:\n        result = normalize_original(archive, folder / 'gtsrb-notebook-results-normalized.zip')\n    else:\n        raise ValueError('Baseline ZIP is not the pinned original or verified export')\n    write_json(folder / 'input_provenance.json', dict(source_notebook=SOURCE, version=VERSION,\n        input_archive_sha256=sha, normalized_archive_sha256=digest(result),\n        verified_content_manifest_sha256=MANIFEST_SHA256, dataset_created=False, api_key_required=False))\n    print('Baseline ZIP ready:', result.name, '| original/verified contents checked | no Dataset or API key', flush=True)\n    return result\n",
  "configs/resnet18.json": "{\n  \"split_seed\": 42,\n  \"validation_fraction\": 0.2,\n  \"crop\": {\n    \"size\": 64,\n    \"padding\": 0.05,\n    \"roi_end_inclusive\": true,\n    \"interpolation\": \"bilinear\"\n  },\n  \"noise_seeds\": [\n    101,\n    202,\n    303\n  ],\n  \"train_seeds\": [\n    11,\n    22,\n    33\n  ],\n  \"conditions\": {\n    \"mild\": {\n      \"gamma_dark\": 1.5,\n      \"poisson_peak\": 120,\n      \"gaussian_sigma\": 0.005\n    },\n    \"medium\": {\n      \"gamma_dark\": 2.2,\n      \"poisson_peak\": 60,\n      \"gaussian_sigma\": 0.01\n    },\n    \"severe\": {\n      \"gamma_dark\": 3.0,\n      \"poisson_peak\": 30,\n      \"gaussian_sigma\": 0.02\n    }\n  },\n  \"cnn\": {\n    \"epochs\": 20,\n    \"patience\": 5,\n    \"min_delta\": 0.001,\n    \"batch_size\": 32,\n    \"learning_rate\": 0.0001,\n    \"normalization\": \"imagenet\",\n    \"architecture\": \"resnet18\",\n    \"weights\": \"IMAGENET1K_V1\",\n    \"input_size\": 224,\n    \"optimizer\": \"adamw\",\n    \"weight_decay\": 0.0001\n  },\n  \"enhancers\": {\n    \"identity\": {},\n    \"gamma\": {\n      \"gamma\": 0.6\n    },\n    \"he\": {},\n    \"clahe\": {\n      \"clip_limit\": 2.0,\n      \"tile_grid\": [\n        8,\n        8\n      ]\n    },\n    \"retinex\": {\n      \"sigmas\": [\n        3,\n        10,\n        20\n      ]\n    },\n    \"zero_dce\": {\n      \"repo\": \"external/Zero-DCE\",\n      \"checkpoint\": \"external/Zero-DCE/Zero-DCE_code/snapshots/Epoch99.pth\",\n      \"pretraining\": \"SICE subset; see author README\"\n    },\n    \"retinexformer\": {\n      \"repo\": \"external/Retinexformer\",\n      \"checkpoint\": \"external/Retinexformer/pretrained_weights/LOL_v1.pth\",\n      \"config\": \"external/Retinexformer/Options/RetinexFormer_LOL_v1.yml\",\n      \"pretraining\": \"LOL-v1\"\n    },\n    \"diffusion\": {\n      \"repo\": \"external/Diffusion-Low-Light\",\n      \"checkpoint\": \"external/Diffusion-Low-Light/ckpt/model.pth.tar\",\n      \"config\": \"external/Diffusion-Low-Light/configs/LOLv1.yml\",\n      \"pretraining\": \"LOLv2 according to author checkpoint metadata; exact subset unverified\",\n      \"sampling_timesteps\": 10,\n      \"inference_seed\": 230,\n      \"sampler\": \"author implicit sampler, eta=0\",\n      \"ema\": true\n    }\n  },\n  \"ssim\": {\n    \"data_range\": 1.0,\n    \"channel_axis\": -1,\n    \"win_size\": 7,\n    \"gaussian_weights\": false,\n    \"use_sample_covariance\": true,\n    \"K1\": 0.01,\n    \"K2\": 0.03\n  },\n  \"bootstrap_repetitions\": 1000,\n  \"selection_tolerance\": 0.005\n}\n",
  "configs/public_sources.json": "{\n  \"dataset\": {\n    \"name\": \"GTSRB official final archives\",\n    \"url\": \"https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/published-archive.html\",\n    \"distribution\": \"Download into temporary runtime storage; do not include source images in notebook outputs.\"\n  },\n  \"models\": [\n    {\n      \"name\": \"zero_dce\",\n      \"sha256\": \"a4395acb874f320375d9704997cef874eaaaaa26a1777ceb29a92b70f74c3612\",\n      \"repository\": \"https://github.com/Li-Chongyi/Zero-DCE.git\",\n      \"commit\": \"e0f4adc54d0f23348c4a9b84acc08fe8778d5bfd\",\n      \"directory\": \"external/Zero-DCE\",\n      \"checkpoint\": \"external/Zero-DCE/Zero-DCE_code/snapshots/Epoch99.pth\",\n      \"download\": {\"kind\": \"url\", \"url\": \"https://raw.githubusercontent.com/Li-Chongyi/Zero-DCE/e0f4adc54d0f23348c4a9b84acc08fe8778d5bfd/Zero-DCE_code/snapshots/Epoch99.pth\"},\n      \"source_page\": \"https://github.com/Li-Chongyi/Zero-DCE\",\n      \"license_note\": \"Author states academic research / CC BY-NC 4.0. Download from author; do not relicense or bundle author weights.\"\n    },\n    {\n      \"name\": \"retinexformer\",\n      \"sha256\": \"9b4810dc51322fab70be8658ee0ad1df64a54aa814497b7849767baa2ff32cd1\",\n      \"repository\": \"https://github.com/caiyuanhao1998/Retinexformer.git\",\n      \"commit\": \"1e9a0efce4b306b6701b824768370ff26066c32a\",\n      \"directory\": \"external/Retinexformer\",\n      \"checkpoint\": \"external/Retinexformer/pretrained_weights/LOL_v1.pth\",\n      \"download\": {\"kind\": \"gdrive\", \"id\": \"1xDwQtTCj3tlAVCTJgYrzonBGVwqeOhKu\"},\n      \"source_page\": \"https://github.com/caiyuanhao1998/Retinexformer#3-testing\",\n      \"license_note\": \"Author repository MIT; retain upstream notices. Download author checkpoint directly.\"\n    },\n    {\n      \"name\": \"diffusion\",\n      \"sha256\": \"acc3b4a41b7a0a12dbd24bbd904d44ddc070cffdcc638dbb038acc3c50715c9d\",\n      \"repository\": \"https://github.com/JianghaiSCU/Diffusion-Low-Light.git\",\n      \"commit\": \"e3223354f47874d3e80d969d3ab2a516874785e8\",\n      \"directory\": \"external/Diffusion-Low-Light\",\n      \"checkpoint\": \"external/Diffusion-Low-Light/ckpt/model.pth.tar\",\n      \"download\": {\"kind\": \"gdrive\", \"id\": \"1f4zDvPsWKrID33OJdeHwc5VOBILkm0KW\"},\n      \"source_page\": \"https://github.com/JianghaiSCU/Diffusion-Low-Light#pre-trained-models\",\n      \"license_note\": \"Author repository MIT; retain upstream notices. Checkpoint pretraining subset remains unverified as noted in experiment config.\"\n    }\n  ]\n}\n",
  "configs/processing_extension.json": "{\n  \"study\": \"denoise_order_extension\",\n  \"validation_per_class\": 30,\n  \"subset_per_class\": 10,\n  \"denoisers\": {\n    \"gaussian\": [{\"kernel\": 3, \"sigma\": 0.5}, {\"kernel\": 5, \"sigma\": 0.8}],\n    \"bilateral\": [{\"diameter\": 5, \"sigma_color\": 0.05, \"sigma_space\": 1.5}, {\"diameter\": 5, \"sigma_color\": 0.1, \"sigma_space\": 3.0}],\n    \"nlm\": [{\"h\": 3, \"h_color\": 3, \"template\": 3, \"search\": 7}, {\"h\": 7, \"h_color\": 7, \"template\": 3, \"search\": 7}]\n  },\n  \"composition_enhancers\": [\"gamma\", \"clahe\"],\n  \"strength_selection\": \"Select each denoiser on validation alone, then fix its parameters across both orders.\",\n  \"fulltest_selection\": \"Validation selects a paired denoiser/enhancer by mean score of both orders; evaluate both orders and their standalone controls. Also retain best standalone denoiser and overall validation winner.\",\n  \"test_selection_forbidden\": true,\n  \"real_lowlight\": false\n}\n"
}
for relative, content in PROJECT_FILES.items():
    path = ROOT / relative
    assert path.resolve().is_relative_to(ROOT.resolve())
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf-8')
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
SOURCE_HASHES = {name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in PROJECT_FILES}
print('Source snapshot:',len(SOURCE_HASHES),'files; no API credentials required.')

## 2. Môi trường và cấu hình

Giữ bộ PyTorch/CUDA của Kaggle. Chỉ cài dependency còn thiếu. Nếu có hai GPU, các training seed và nhóm đánh giá được phân công độc lập giữa hai GPU; mỗi seed vẫn giữ batch size, optimizer và dữ liệu như cấu hình. Trọng số ba enhancer chỉ dùng suy luận.

Full-test tốn nhiều giờ. Notebook giải phóng cache sau khi cả ba classifier dùng xong một nhóm ảnh; không lưu toàn bộ cache vào Output. Không tự giảm số seed hoặc phương pháp khi sắp hết thời gian.

In [ ]:
import importlib.util, importlib.metadata
dependencies = {'einops':'einops','yaml':'PyYAML','pytorch_msssim':'pytorch-msssim','gdown':'gdown',
                'skimage':'scikit-image','sklearn':'scikit-learn','cv2':'opencv-python-headless'}
missing = [package for module,package in dependencies.items() if importlib.util.find_spec(module) is None]
try:
    importlib.metadata.version('opencv-python-headless')
except importlib.metadata.PackageNotFoundError:
    missing.append('opencv-python-headless')
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*dict.fromkeys(missing)],check=True)
import torch, numpy as np, pandas as pd
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Bật GPU và Internet trong Settings trước khi chạy.'
DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())]
CONFIG = json.loads((ROOT/'configs/resnet18.json').read_text())
TRAIN_MODE = 'reuse'  # 'fresh' để chạy toàn bộ từ đầu.
CHECKPOINT_ARCHIVE = None  # Optional: ZIP đã có trong phiên; None tải trực tiếp Output v1.
REQUIRE_BASELINE_REUSE = True  # Stop instead of rerunning completed baseline experiments.
RUN_ABLATION = TRAIN_MODE == 'fresh'
RUN_EXTENSION_FULL_TEST = True
assert TRAIN_MODE in {'fresh','reuse'}
if TRAIN_MODE == 'reuse':
    from src.tools.baseline_input import resolve_baseline
    CHECKPOINT_ARCHIVE = resolve_baseline(ROOT/'baseline_input',CHECKPOINT_ARCHIVE)
OUT = ROOT/('outputs/fresh' if TRAIN_MODE=='fresh' else 'outputs/reuse')
OUT.mkdir(parents=True,exist_ok=True)
NOTEBOOK_URL = 'https://www.kaggle.com/code/kieuthithutrang/gtsrb-low-light-enhancement-for-traffic-signs'
from src.common import write_json, environment
def status(stage, **extra):
    record = {'stage':stage,'notebook_url':NOTEBOOK_URL,'fresh_training':TRAIN_MODE=='fresh','extension_requested':True,**extra}
    write_json(PUBLIC/'RUN_STATUS.json',record)
    print('STAGE:',stage,flush=True)
status('environment_ready',devices=DEVICES)
print('PyTorch:',torch.__version__,'GPU:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 3. Tải và kiểm tra trọng số tác giả trước khi train

Kiểm tra tải được file, đúng checksum, đúng kiến trúc và suy luận được RGB 64×64. Nếu nguồn tác giả không truy cập được, dừng trước huấn luyện; không thay bằng trọng số khác hoặc lấy từ dataset private.

In [ ]:
from src.tools.public_sources import provision
from src.enhancement import Enhancer
UPSTREAM = provision(ROOT,ROOT/'configs/public_sources.json')
for method in ['zero_dce','retinexformer','diffusion']:
    enhancer = Enhancer(method,CONFIG['enhancers'][method],torch.device(DEVICES[0]))
    preview = enhancer(np.full((64,64,3),.15,dtype=np.float32),'preflight')
    assert preview.shape == (64,64,3) and np.isfinite(preview).all()
    print('Forward OK:',method,flush=True)
    del enhancer
torch.cuda.empty_cache()
write_json(OUT/'upstream_sources.json',UPSTREAM)
write_json(OUT/'source_hashes.json',SOURCE_HASHES)
status('upstream_verified')

## 4. GTSRB chính thức và chia dữ liệu

Nhãn train nằm trong 43 file `GT-*.csv` ngay trong ZIP ảnh train; nhãn test tải bằng gói GT test riêng. Chia train/validation theo class+track, seed 42, cách ly track có ảnh trùng pixel với test; crop ROI padding 5%, RGB 64×64. Chọn trước ID subset test để không chọn ví dụ theo kết quả.

In [ ]:
from src.data import download, prepare, audit, subset_rows
from src.common import read_csv
download(ROOT)
prepare(ROOT,CONFIG)
audit(ROOT)
rows = read_csv(ROOT/'data/manifests/gtsrb_samples.csv')
display(pd.DataFrame(rows).groupby('split').size().rename('Số ảnh').to_frame())
assert sum(r['split']=='test' for r in rows) == 12630
assert sum(r['split']=='train' for r in rows) == 31350
assert sum(r['split']=='validation' for r in rows) == 7829
write_json(OUT/'predeclared_subset_ids.json',[r['sample_id'] for r in subset_rows(rows,'test',10,42)])
status('dataset_verified')

## 5. Khôi phục hoặc huấn luyện ba ResNet-18

`reuse`: tải ZIP trực tiếp từ Output của phiên v1 của chính notebook, không tạo Dataset và không cần API key. Có thể điền CHECKPOINT_ARCHIVE để dùng ZIP đã có. Kiểm tra checksum gói gốc, sửa riêng đường dẫn supplemental bị trùng và đối chiếu toàn bộ manifest với gói đã kiểm chứng; không thay dự đoán hay checkpoint. Sau đó kiểm tra ba seed, manifest và cấu hình CNN; chỉ đọc checkpoint bằng weights_only, không chạy mã trong ZIP. Selection và dự đoán chỉ được dùng lại khi cùng mã, dữ liệu và checkpoint. `fresh`: ImageNet, tinh chỉnh ảnh gốc, seed 11/22/33, tối đa 20 epoch; chọn checkpoint bằng Macro-F1 validation clean. Sao lưu checkpoint ngay sau bước này.

In [ ]:
from src.tools.fresh_run import train_fresh
from src.common import digest
if TRAIN_MODE == 'fresh':
    status('training',devices=DEVICES)
    CHECKPOINTS = train_fresh(ROOT,CONFIG,OUT/'train',DEVICES)
else:
    from src.tools.checkpoint_restore import restore
    CHECKPOINTS, RESTORED = restore(ROOT,CHECKPOINT_ARCHIVE,OUT,CONFIG)
    shutil.copy2(ROOT/'baseline_input/input_provenance.json',OUT/'train/input_provenance.json')
    print(RESTORED)
    if REQUIRE_BASELINE_REUSE:
        assert RESTORED['selection_reusable'] and set(RESTORED['baseline_scopes'])=={'subset','fulltest'} and RESTORED['ablation_reusable'], 'Baseline artifacts cannot be reused exactly; stopping before any baseline retraining or reevaluation.'
write_json(OUT/'checkpoint_hashes.json',{p.name:digest(p) for p in CHECKPOINTS})
import zipfile
backup = PUBLIC/'gtsrb-checkpoints.zip'
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    hashes={}
    for p in CHECKPOINTS:
        name='05_training/'+p.name;z.write(p,name);hashes[name]=digest(p)
    z.writestr('SHA256.json',json.dumps(hashes,indent=2))
status('classifiers_ready',checkpoint_backup=backup.name)

## 6. Chọn phương pháp trên validation

1.290 ảnh validation, 3 mức suy giảm và 3 noise seed, cùng ba classifier. Chọn cấu hình Gamma/CLAHE/MSR và pipeline bằng Macro-F1 validation; chưa xem test. Cache giải phóng sau khi ba checkpoint dùng xong.

In [ ]:
from src.experiment import tune, freeze
status('validation_selection')
if (OUT/'validation/selection.json').exists():
    selection = json.loads((OUT/'validation/selection.json').read_text())
else:
    selection = tune(ROOT,CONFIG,CHECKPOINTS,torch.device(DEVICES[0]),OUT/'validation',per_class=30,release_cache=True)
torch.cuda.empty_cache()
display(pd.DataFrame(selection['scores']))
print('Pipeline chọn trước test:',selection['chosen_method'])
status('validation_completed',selected_method=selection['chosen_method'])

## 7. Subset 430 ảnh: kiểm tra giao thức và phân tích lỗi

219 lượt, đủ 43 lớp (10 ảnh/lớp). Tính lại metric từ dự đoán, F1 từng lớp, ma trận nhầm lẫn và các trường hợp được giúp/bị làm sai. Đây là subset; không gọi là full-test. Xuất gói subset trước để vẫn giữ được bằng chứng nếu phiên Kaggle bị ngắt trong các bước dài sau đó.

In [ ]:
from src.tools.fresh_run import parallel_evaluate, ablation
from src.tools.research_evidence import analyze, audited_runs
from src.tools.result_archive import pack, aggregate
subset_protocol = OUT/'protocol_subset.json'
status('subset_evaluation')
if (OUT/'subset/summary.csv').exists():
    audited_runs(OUT/'subset')
else:
    if not subset_protocol.exists():freeze(ROOT,OUT/'validation/selection.json',subset_protocol,per_class=10)
    parallel_evaluate(ROOT,subset_protocol,OUT/'subset',DEVICES)
if not (OUT/'subset/error_analysis/audit.json').exists():analyze(OUT/'subset',OUT/'subset/error_analysis')
display(pd.DataFrame(aggregate(read_csv(OUT/'subset/summary.csv')))[['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd']])
if not (PUBLIC/'gtsrb-subset-results.zip').exists():pack(OUT/'subset',PUBLIC/'gtsrb-subset-results.zip',analysis=OUT/'subset/error_analysis',source=ROOT)
status('subset_completed',prediction_rows=94170)

## 8. Ablation trên subset

Tách Gamma-only, Gamma+Gaussian, Gamma+Poisson và Gamma+cả hai, giữ nguyên checkpoint và tham số đã chọn. Có 288 lượt. Không dùng kết quả ablation để chọn lại pipeline.

In [ ]:
if RUN_ABLATION:
    status('ablation')
    ablation(ROOT,CONFIG,selection,CHECKPOINTS,torch.device(DEVICES[0]),OUT/'ablation')
    display(pd.DataFrame(read_csv(OUT/'ablation/summary.csv')).head())
else:
    print('Dùng lại ablation đã kiểm tra.' if (OUT/'ablation/summary.csv').exists() else 'Ablation không được yêu cầu.')
torch.cuda.empty_cache()
status('ablation_completed' if RUN_ABLATION else 'ablation_not_requested')

## 9. Full-test chính thức 12.630 ảnh

Khóa cùng selection rồi đánh giá đủ 219 lượt, tổng cộng 2.765.970 dự đoán. Các nhóm condition/noise/method độc lập được phân cho GPU khả dụng; cả ba checkpoint nhận cùng ảnh tăng cường trong mỗi nhóm. Cache được tái tạo đúng seed và giải phóng sau nhóm, giữ thông tin hash và dự đoán. Nếu chưa xong bước này, không coi bảng subset là full-test.

In [ ]:
full_protocol = OUT/'protocol_fulltest.json'
status('fulltest_evaluation')
if (OUT/'fulltest/summary.csv').exists():
    audited_runs(OUT/'fulltest')
else:
    if not full_protocol.exists():freeze(ROOT,OUT/'validation/selection.json',full_protocol)
    parallel_evaluate(ROOT,full_protocol,OUT/'fulltest',DEVICES)
if not (OUT/'fulltest/error_analysis/audit.json').exists():analyze(OUT/'fulltest',OUT/'fulltest/error_analysis')
full_aggregate = pd.DataFrame(aggregate(read_csv(OUT/'fulltest/summary.csv')))
display(full_aggregate[['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd']])
status('fulltest_verified',runs=219,images_per_run=12630,prediction_rows=2765970)
baseline_backup = PUBLIC/'gtsrb-baseline-results.zip'
if not baseline_backup.exists():
    pack(OUT/'fulltest',baseline_backup,analysis=OUT/'fulltest/error_analysis',source=ROOT,
         supplemental=OUT/'ablation',subset=OUT/'subset')
print('Baseline full-test đã được sao lưu trước khi chạy mở rộng:',baseline_backup.name)

## 10. Mở rộng: khử nhiễu và thứ tự xử lý

Gaussian, bilateral và Non-local Means: hai cấu hình mỗi bộ lọc. Chọn độ mạnh trên 1.290 ảnh validation rồi **giữ cùng tham số** khi so hai thứ tự với Gamma và CLAHE. Tổng cộng 6 ứng viên độc lập và 12 chuỗi kết hợp cần đánh giá thêm trên validation. Không thay CNN, mức tối, noise seed hoặc test ID.

Subset chạy mọi họ phương pháp. Validation chọn một cặp thứ tự theo điểm trung bình hai thứ tự, kèm đối chứng độc lập, bộ lọc tốt nhất và ứng viên tổng thể tốt nhất cho full-test. Khóa cả hai giao thức trước khi đọc test; không chọn theo subset test. NLM chuyển RGB sang uint8 BGR theo OpenCV và ghi lượng tử hóa vào provenance. Pipeline không đọc nhãn hay ảnh gốc; bộ đánh giá mới đọc để tính metric.

In [ ]:
from src.tools import processing_study as extension
EXT = OUT/'processing_extension'
status('extension_validation')
extension_selection = extension.tune(ROOT,OUT/'validation/selection.json',CHECKPOINTS,
    ROOT/'configs/processing_extension.json',torch.device(DEVICES[0]),EXT/'validation')
display(pd.DataFrame(extension_selection['scores'])[['method','category','macro_f1','latency_ms']])
print('Full-test methods locked from validation:',extension_selection['fulltest_methods'])
extension.freeze(ROOT,EXT/'validation/selection.json',EXT/'protocol_subset.json',per_class=10)
extension.freeze(ROOT,EXT/'validation/selection.json',EXT/'protocol_fulltest.json',per_class=None)
for scope in (['subset','fulltest'] if RUN_EXTENSION_FULL_TEST else ['subset']):
    status('extension_'+scope)
    extension.evaluate(ROOT,EXT/('protocol_'+scope+'.json'),EXT/scope,torch.device(DEVICES[0]),
        baseline_output=OUT/scope)
    if not (EXT/scope/'error_analysis/audit.json').exists():
        extension.analyze(EXT/scope,EXT/scope/'error_analysis')
    display(pd.DataFrame(aggregate(read_csv(EXT/scope/'summary.csv')))[
        ['condition','method','accuracy_mean','macro_f1_mean','macro_f1_sd','psnr_mean_mean','ssim_mean_mean']])
    if scope=='subset' and not (PUBLIC/'gtsrb-processing-subset-results.zip').exists():
        pack(EXT/scope,PUBLIC/'gtsrb-processing-subset-results.zip',
             analysis=EXT/scope/'error_analysis',include_checkpoints=False)
torch.cuda.empty_cache()
status('extension_verified',extension_fulltest_completed=RUN_EXTENSION_FULL_TEST)

## 11. Xuất gói kết quả dễ đọc

Mở `READ_ME_FIRST.md` trong ZIP trước. Bảng dự đoán test gộp vào một CSV với run_id/method/condition/seed; confusion matrix gộp vào NPZ, có bản kê SHA-256. Validation, subset và ablation nằm riêng, không trộn với full-test. Không đóng gói ảnh gốc, cache hoặc trọng số enhancer tác giả.

Gói subset có checkpoint để khôi phục nếu phiên bị ngắt. Sau khi full-test hoàn tất, checkpoint chỉ nằm trong gói đầy đủ, tránh giữ hai bản trùng nhau. Kết quả ảnh đêm thực chưa được tạo vì chưa có dữ liệu thực gán nhãn tương thích.

In [ ]:
final_zip = PUBLIC/'gtsrb-notebook-results.zip'
staged_zip = PUBLIC/'.gtsrb-final-export.zip'
if staged_zip.exists():staged_zip.unlink()
pack(OUT/'fulltest',staged_zip,analysis=OUT/'fulltest/error_analysis',source=ROOT,
     supplemental=OUT/'ablation',subset=OUT/'subset',extension=EXT)
staged_zip.replace(final_zip)
import zipfile
with zipfile.ZipFile(final_zip) as z:
    assert z.testzip() is None
    print('ZIP:',final_zip.name,'files:',len(z.namelist()),'MB:',round(final_zip.stat().st_size/1e6,1))
    print('Top-level:',sorted({n.split('/')[0] for n in z.namelist()}))
# Replace only our own explicitly named subset ZIP with a tables-only version.
tables_only = PUBLIC/'.gtsrb-subset-tables.zip'
if tables_only.exists():tables_only.unlink()
pack(OUT/'subset',tables_only,analysis=OUT/'subset/error_analysis',include_checkpoints=False)
tables_only.replace(PUBLIC/'gtsrb-subset-results.zip')
# Remove our explicitly named interim backups only after final ZIP verification.
for name in ['gtsrb-checkpoints.zip','gtsrb-baseline-results.zip','gtsrb-processing-subset-results.zip']:
    path = PUBLIC/name
    if path.exists():path.unlink()
status('completed',final_archive=final_zip.name,archive_sha256=digest(final_zip),
       fulltest_prediction_rows=2765970,real_lowlight_evaluated=False,extension_fulltest_completed=RUN_EXTENSION_FULL_TEST)
print('COMPLETE — tải gtsrb-notebook-results.zip ở Output của notebook.')

## Tài liệu và phạm vi

- GTSRB: J. Stallkamp, M. Schlipsing, J. Salmen, C. Igel, *The German Traffic Sign Recognition Benchmark: A multi-class classification competition*, IJCNN 2011.
- Zero-DCE: C. Guo et al., CVPR 2020.
- Retinexformer: Y. Cai et al., ICCV 2023.
- Wavelet-based Diffusion LLIE: H. Jiang et al., ACM TOG 2023.

Ảnh crop và suy giảm tổng hợp phục vụ nghiên cứu. PSNR/SSIM đo so ảnh gốc trên cùng crop; không thay thế hiệu năng phân loại. Độ lệch chuẩn qua ba checkpoint không phải khoảng tin cậy cho camera thực. Không có phát hiện biển báo toàn cảnh hay đánh giá ảnh đêm thực trong lần chạy này.